# Dataset profiling

In [4]:
# ============================================================
# PREP-0 — DATASET PROFILING (ONE CELL, FULL) — REVISI FULL v3.3
#
# Tujuan:
# - Index train/test/supplemental + mask mapping yang robust
# - Profiling IMAGE (train + test): size, is_gray, bg_white_frac, ROI box, ROI area frac, edge_density
# - Profiling MASK (train forged yang punya masks): union area_frac, n_components, mismatch vs image, mask_outside_roi_frac
# - Case summary per case_id (has_auth/has_forg/has_supp, max_n_masks, n_rows)
# - Cache artifact robust + auto-invalidate cache yang rusak / dataset berubah
# - Output terstruktur untuk tahap selanjutnya:
#   * PATHS, ART_DIR
#   * df_train_all (punya sample_id, fold, roi_*, bg_white_frac, edge_density, gt_* jika ada)
#   * df_test + df_test_profile (buat ROI lookup tahap DINO cache)
#   * df_mask_profile, df_img_profile_train, df_img_profile_test, df_case_summary
#   * profile_summary
#
# Catatan kompatibilitas:
# - "mask_paths" disimpan sebagai LIST di memori untuk dipakai langsung.
# - Untuk penyimpanan parquet, juga dibuat "mask_paths_json" (lebih aman).
# ============================================================

import os, re, json, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# 0) (Opsional) Path model untuk tahap berikutnya (disimpan saja)
# ----------------------------
DINO_MODEL_DIR = Path("/kaggle/input/m/keras/dinov2/keras/dinov2_giant/1")
DINO_DIR = Path("/kaggle/input/dinov2/pytorch/giant/1")  # dipakai di STAGE 2 versi HF Torch kamu
print("DINO_MODEL_DIR:", DINO_MODEL_DIR, "| exists:", DINO_MODEL_DIR.exists())
print("DINO_DIR      :", DINO_DIR, "| exists:", DINO_DIR.exists())

# ----------------------------
# 1) Locate competition dataset root
# ----------------------------
INP = Path("/kaggle/input")

def find_comp_root():
    pref = []
    for d in INP.iterdir():
        if not d.is_dir():
            continue
        name = d.name.lower()
        ok = (d / "train_images").exists() and (d / "test_images").exists() and (d / "sample_submission.csv").exists()
        if ok and ("recodai" in name or "luc" in name or "forgery" in name):
            pref.append(d)
    if pref:
        pref = sorted(pref, key=lambda p: len(p.parts))
        return pref[0]

    cands = []
    for d in INP.iterdir():
        if not d.is_dir():
            continue
        ok = (d / "train_images").exists() and (d / "test_images").exists() and (d / "sample_submission.csv").exists()
        if ok:
            cands.append(d)

    if not cands:
        for ss in INP.rglob("sample_submission.csv"):
            root = ss.parent
            if (root / "train_images").exists() and (root / "test_images").exists():
                cands.append(root)

    if not cands:
        raise FileNotFoundError("Tidak menemukan dataset root dengan train_images/test_images/sample_submission.csv di /kaggle/input")

    cands = sorted(cands, key=lambda p: len(p.parts))
    return cands[0]

COMP_ROOT = find_comp_root()
print("\nCOMP_ROOT:", COMP_ROOT)

# ----------------------------
# 2) Build PATHS dictionary
# ----------------------------
PATHS = {
    "ROOT": COMP_ROOT,
    "SAMPLE_SUB": COMP_ROOT / "sample_submission.csv",
    "TRAIN_IMAGES": COMP_ROOT / "train_images",
    "TEST_IMAGES":  COMP_ROOT / "test_images",
    "TRAIN_MASKS":  COMP_ROOT / "train_masks",
    "SUPP_IMAGES":  COMP_ROOT / "supplemental_images",
    "SUPP_MASKS":   COMP_ROOT / "supplemental_masks",
    "TRAIN_AUTH":   (COMP_ROOT / "train_images" / "authentic"),
    "TRAIN_FORG":   (COMP_ROOT / "train_images" / "forged"),
}

print("\nPATHS check:")
for k in ["TRAIN_AUTH","TRAIN_FORG","TEST_IMAGES","TRAIN_MASKS","SUPP_IMAGES","SUPP_MASKS","SAMPLE_SUB"]:
    p = PATHS.get(k)
    print(f" - {k:12s}: {str(p)} | exists={p.exists() if p is not None else False}")

# ----------------------------
# 3) Artifacts dir (+ version signature)
# ----------------------------
ART_DIR = Path("/kaggle/working/recodai_luc/artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)
print("\nART_DIR:", ART_DIR)

# Signature untuk invalidasi cache saat dataset berubah
_sig = {
    "comp_root": str(COMP_ROOT),
    "has_supp_images": bool(PATHS["SUPP_IMAGES"].exists()),
    "has_supp_masks": bool(PATHS["SUPP_MASKS"].exists()),
    "train_masks_dir": str(PATHS["TRAIN_MASKS"]),
}
SIG_ID = hashlib.sha1(json.dumps(_sig, sort_keys=True).encode("utf-8")).hexdigest()[:10]
META_PATH = ART_DIR / "profile_meta.json"
print("SIG_ID:", SIG_ID)

def _write_meta(extra: dict = None):
    meta = {"sig_id": SIG_ID, "sig": _sig, "version": "PREP-0_v3.3"}
    if extra:
        meta.update(extra)
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

def _meta_matches() -> bool:
    if not META_PATH.exists():
        return False
    try:
        meta = json.load(open(META_PATH, "r", encoding="utf-8"))
        return str(meta.get("sig_id","")) == SIG_ID
    except Exception:
        return False

# ----------------------------
# 4) Helpers: parse case_id, list images, index masks
# ----------------------------
IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}

def parse_case_id_from_name(name: str):
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else None

def list_images(folder: Path):
    if folder is None or (not folder.exists()):
        return []
    out = []
    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            cid = parse_case_id_from_name(p.stem)
            if cid is not None:
                out.append((cid, p))
    return out

def index_masks(mask_dir: Path):
    mask_map = {}
    if mask_dir is None or (not mask_dir.exists()):
        return mask_map
    for p in mask_dir.iterdir():
        if not p.is_file():
            continue
        suf = p.suffix.lower()
        if suf not in {".npy", ".npz", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}:
            continue
        cid = parse_case_id_from_name(p.stem)
        if cid is None:
            continue
        mask_map.setdefault(cid, []).append(p)
    for cid in mask_map:
        mask_map[cid] = sorted(mask_map[cid])
    return mask_map

mask_map_train = index_masks(PATHS["TRAIN_MASKS"])
mask_map_supp  = index_masks(PATHS["SUPP_MASKS"])

# ----------------------------
# 5) Build df_train_all (dengan sample_id & has_mask)
# ----------------------------
rows = []

# train authentic
for cid, p in list_images(PATHS["TRAIN_AUTH"]):
    rows.append({
        "case_id": int(cid),
        "variant": "train_authentic",
        "source": "train",
        "y_forged": 0,
        "image_path": str(p),
        "mask_paths": [],
        "n_masks": 0,
        "has_mask": 0,
    })

# train forged
for cid, p in list_images(PATHS["TRAIN_FORG"]):
    mps = mask_map_train.get(int(cid), [])
    rows.append({
        "case_id": int(cid),
        "variant": "train_forged",
        "source": "train",
        "y_forged": 1,
        "image_path": str(p),
        "mask_paths": [str(x) for x in mps],
        "n_masks": int(len(mps)),
        "has_mask": int(len(mps) > 0),
    })

# supplemental (label mungkin tidak lengkap)
if PATHS["SUPP_IMAGES"].exists():
    for cid, p in list_images(PATHS["SUPP_IMAGES"]):
        mps = mask_map_supp.get(int(cid), [])
        # y_forged: 1 kalau ada mask, -1 jika tidak ada (unknown / unlabeled)
        y = 1 if len(mps) > 0 else -1
        rows.append({
            "case_id": int(cid),
            "variant": "supplemental",
            "source": "supplemental",
            "y_forged": int(y),
            "image_path": str(p),
            "mask_paths": [str(x) for x in mps],
            "n_masks": int(len(mps)),
            "has_mask": int(len(mps) > 0),
        })

df_train_all = pd.DataFrame(rows)
df_train_all = df_train_all.sort_values(["source","case_id","variant"]).reset_index(drop=True)

# sample_id unik per baris (kompatibel STAGE 2 yang pakai ROI_LOOKUP by sample_id)
df_train_all["sample_id"] = df_train_all.apply(lambda r: f"{int(r['case_id'])}_{str(r['variant'])}", axis=1)

n_total = len(df_train_all)
n_auth  = int((df_train_all["y_forged"]==0).sum())
n_forg  = int((df_train_all["y_forged"]==1).sum())
n_supp  = int((df_train_all["source"]=="supplemental").sum())
print("\nTRAIN INDEX:")
print(" - rows:", n_total, "| authentic:", n_auth, "| forged:", n_forg, "| supplemental:", n_supp)

df_forg = df_train_all[df_train_all["y_forged"]==1].copy()
n_missing_masks = int((df_forg["has_mask"]==0).sum()) if len(df_forg) else 0
print(" - forged with missing masks:", n_missing_masks, "of", len(df_forg))

# ----------------------------
# 6) Build df_test in correct order (from sample_submission.csv)
# ----------------------------
df_sub = pd.read_csv(PATHS["SAMPLE_SUB"])
if "case_id" in df_sub.columns:
    SUB_ID_COL = "case_id"
elif "id" in df_sub.columns:
    SUB_ID_COL = "id"
else:
    raise ValueError(f"sample_submission columns not recognized. Found: {list(df_sub.columns)}")

df_sub[SUB_ID_COL] = pd.to_numeric(df_sub[SUB_ID_COL], errors="coerce").astype("Int64")
if df_sub[SUB_ID_COL].isna().any():
    raise ValueError("sample_submission has non-numeric ids. Cannot map to test_images.")

test_files = {int(cid): p for cid, p in list_images(PATHS["TEST_IMAGES"])}

df_test = pd.DataFrame({"case_id": df_sub[SUB_ID_COL].astype(int).values})
df_test["variant"] = "test"
df_test["source"] = "test"
df_test["y_forged"] = -1
df_test["image_path"] = df_test["case_id"].map(lambda x: str(test_files.get(int(x), "")))
df_test["has_file"] = df_test["image_path"].map(lambda s: (s is not None) and (len(str(s)) > 0))

missing_test = int((~df_test["has_file"]).sum())
print("\nTEST INDEX:")
print(" - rows:", len(df_test), "| missing files (by case_id mapping):", missing_test)

# ----------------------------
# 6b) Fold assignment (GroupKFold by case_id, leakage-safe)
#     Semua row untuk case_id yang sama akan dapat fold sama.
# ----------------------------
N_FOLDS = 5
try:
    from sklearn.model_selection import GroupKFold
    gkf = GroupKFold(n_splits=N_FOLDS)
    # gunakan semua train_all (termasuk supplemental) untuk grouping, tapi split berdasarkan case_id unik
    df_train_all["fold"] = -1
    unique_cases = df_train_all[["case_id"]].drop_duplicates().reset_index(drop=True)
    dummy_X = np.zeros((len(unique_cases), 1), dtype=np.float32)
    groups = unique_cases["case_id"].values
    for f, (_, va_idx) in enumerate(gkf.split(dummy_X, dummy_X, groups=groups)):
        va_cases = set(groups[va_idx].tolist())
        df_train_all.loc[df_train_all["case_id"].isin(va_cases), "fold"] = int(f)
    print("\nCV:")
    print(" - folds:", N_FOLDS, "| fold value counts:", df_train_all["fold"].value_counts(dropna=False).to_dict())
except Exception as e:
    df_train_all["fold"] = -1
    print("\nCV: fallback (no sklearn). fold set to -1. Err:", type(e).__name__, str(e))

# ----------------------------
# 7) Mask I/O: robust normalize to 2D binary mask
# ----------------------------
def load_mask_raw(path: str):
    p = Path(path)
    suf = p.suffix.lower()
    if suf == ".npy":
        return np.load(p, allow_pickle=False)
    if suf == ".npz":
        z = np.load(p, allow_pickle=False)
        key = "arr_0" if "arr_0" in z.files else z.files[0]
        return z[key]
    from PIL import Image
    return np.array(Image.open(p))

def to_2d_binary_mask(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 0:
        return np.zeros((0,0), dtype=np.uint8)

    while a.ndim > 2:
        if a.ndim == 3:
            # (K,H,W) => union axis 0
            if (a.shape[0] <= 32) and (a.shape[1] > 16) and (a.shape[2] > 16):
                a = a.max(axis=0)
                continue
            # (H,W,C) => union channel
            if (a.shape[2] <= 8) and (a.shape[0] > 16) and (a.shape[1] > 16):
                a = a.max(axis=2)
                continue
            a = np.squeeze(a)
            if a.ndim == 3:
                k = int(np.argmin(list(a.shape)))
                a = a.max(axis=k)
            continue

        if a.ndim == 4:
            a = np.squeeze(a)
            if a.ndim == 4:
                if a.shape[0] <= 32:
                    a = a.max(axis=0)
                else:
                    a = a.reshape(a.shape[-3], a.shape[-2], a.shape[-1])
            continue

        a = np.squeeze(a)
        if a.ndim > 2:
            a = a.reshape(a.shape[-2], a.shape[-1])

    if a.ndim != 2:
        a = np.zeros((0,0), dtype=np.uint8)

    return (a > 0).astype(np.uint8)

def read_mask_any(path: str) -> np.ndarray:
    raw = load_mask_raw(path)
    return to_2d_binary_mask(raw)

# ----------------------------
# 8) Connected components helper
# ----------------------------
_cc_backend = None
try:
    import cv2
    _cc_backend = "cv2"
except Exception:
    try:
        from scipy.ndimage import label as scipy_label
        _cc_backend = "scipy"
    except Exception:
        _cc_backend = None

def count_components(bin_mask: np.ndarray) -> int:
    if bin_mask is None or bin_mask.size == 0:
        return 0
    if int(bin_mask.sum()) == 0:
        return 0
    if _cc_backend == "cv2":
        n, _ = cv2.connectedComponents(bin_mask.astype(np.uint8), connectivity=8)
        return int(n - 1)
    if _cc_backend == "scipy":
        _, n = scipy_label(bin_mask.astype(np.uint8))
        return int(n)
    return 1

# ----------------------------
# 9) Image profiling (train + test) — cached + invalidate by SIG_ID
# ----------------------------
img_profile_train_path = ART_DIR / "image_profile_train.parquet"
img_profile_test_path  = ART_DIR / "image_profile_test.parquet"

def _df_cache_ok(df: pd.DataFrame, need_cols: set) -> bool:
    if df is None or len(df) == 0:
        return False
    if not need_cols.issubset(set(df.columns)):
        return False
    try:
        med_h = float(pd.to_numeric(df["img_H"], errors="coerce").dropna().median())
        med_w = float(pd.to_numeric(df["img_W"], errors="coerce").dropna().median())
        if (not np.isfinite(med_h)) or (not np.isfinite(med_w)) or med_h < 32 or med_w < 32:
            return False
    except Exception:
        return False
    return True

def compute_image_quick_stats(img_path: str, max_side: int = 512, bg_thr: int = 250,
                              min_valid_frac: float = 0.05, margin_px: int = 8, edge_thr: float = 20.0):
    from PIL import Image

    with Image.open(img_path) as im:
        im = im.convert("RGB")
        W, H = im.size

        im_small = im.copy()
        im_small.thumbnail((max_side, max_side), resample=Image.BILINEAR)
        arr = np.array(im_small, dtype=np.uint8)

    hs, ws = arr.shape[:2]
    g = arr.mean(axis=2).astype(np.float32)

    # is_gray (approx)
    rg = np.abs(arr[...,0].astype(np.int16) - arr[...,1].astype(np.int16))
    gb = np.abs(arr[...,1].astype(np.int16) - arr[...,2].astype(np.int16))
    is_gray = float((rg.mean() + gb.mean()) / 2.0 < 1.5)

    bg_white_frac = float((g >= bg_thr).mean())

    valid = (g < bg_thr)
    ys, xs = np.where(valid)
    if len(xs) < int(min_valid_frac * hs * ws):
        x0s, y0s, x1s, y1s = 0, 0, ws, hs
    else:
        x0s, x1s = int(xs.min()), int(xs.max())
        y0s, y1s = int(ys.min()), int(ys.max())
        msx = max(2, int(margin_px * ws / max(W, 1)))
        msy = max(2, int(margin_px * hs / max(H, 1)))
        x0s = max(0, x0s - msx); y0s = max(0, y0s - msy)
        x1s = min(ws, x1s + msx); y1s = min(hs, y1s + msy)

    x0 = int(round(x0s * W / max(ws, 1)))
    x1 = int(round(x1s * W / max(ws, 1)))
    y0 = int(round(y0s * H / max(hs, 1)))
    y1 = int(round(y1s * H / max(hs, 1)))
    x0 = max(0, min(x0, W)); x1 = max(0, min(x1, W))
    y0 = max(0, min(y0, H)); y1 = max(0, min(y1, H))
    if (x1 <= x0 + 4) or (y1 <= y0 + 4):
        x0, y0, x1, y1 = 0, 0, W, H

    roi_area_frac = float(((x1 - x0) * (y1 - y0)) / max(W * H, 1))

    dx = np.abs(np.diff(g, axis=1))
    dy = np.abs(np.diff(g, axis=0))
    dxp = np.pad(dx, ((0,0),(0,1)), mode="edge")
    dyp = np.pad(dy, ((0,1),(0,0)), mode="edge")
    edge_density = float(((dxp + dyp) > edge_thr).mean())

    return {
        "img_H": int(H),
        "img_W": int(W),
        "aspect": float(W / max(H, 1)),
        "is_gray": float(is_gray),
        "bg_white_frac": float(bg_white_frac),
        "roi_x0": int(x0), "roi_y0": int(y0), "roi_x1": int(x1), "roi_y1": int(y1),
        "roi_area_frac": float(roi_area_frac),
        "edge_density": float(edge_density),
    }

need_img_cols = {"case_id","variant","img_H","img_W","bg_white_frac","roi_area_frac","edge_density","is_gray","roi_x0","roi_y0","roi_x1","roi_y1"}

df_img_profile_train = None
df_img_profile_test  = None

# Load caches only if META matches current dataset signature
if _meta_matches():
    if img_profile_train_path.exists():
        try:
            tmp = pd.read_parquet(img_profile_train_path)
            if _df_cache_ok(tmp, need_img_cols):
                df_img_profile_train = tmp
                print("\nLoaded cached image_profile_train:", img_profile_train_path, "| rows:", len(df_img_profile_train))
        except Exception:
            df_img_profile_train = None
    if img_profile_test_path.exists():
        try:
            tmp = pd.read_parquet(img_profile_test_path)
            if _df_cache_ok(tmp, need_img_cols):
                df_img_profile_test = tmp
                print("Loaded cached image_profile_test :", img_profile_test_path,  "| rows:", len(df_img_profile_test))
        except Exception:
            df_img_profile_test = None
else:
    print("\nCache meta mismatch / not found -> recompute profiles.")

def _compute_img_profile(df: pd.DataFrame, label: str):
    try:
        from tqdm.auto import tqdm
    except Exception:
        tqdm = lambda x, **kw: x

    print(f"\nComputing IMAGE profile ({label}) on rows:", len(df))
    img_rows, fail = [], 0
    for r in tqdm(df.itertuples(index=False), total=len(df)):
        cid = int(getattr(r, "case_id"))
        variant = str(getattr(r, "variant", ""))
        src = str(getattr(r, "source", ""))
        ip = str(getattr(r, "image_path", ""))
        try:
            st = compute_image_quick_stats(ip)
            st.update({"case_id": cid, "variant": variant, "source": src, "image_path": ip})
            img_rows.append(st)
        except Exception:
            fail += 1
            img_rows.append({
                "case_id": cid, "variant": variant, "source": src, "image_path": ip,
                "img_H": -1, "img_W": -1, "aspect": np.nan, "is_gray": np.nan,
                "bg_white_frac": np.nan, "roi_x0": 0, "roi_y0": 0, "roi_x1": 0, "roi_y1": 0,
                "roi_area_frac": np.nan, "edge_density": np.nan
            })
    out = pd.DataFrame(img_rows)
    print(f"Done IMAGE profile ({label}) | fails:", fail)
    return out

if df_img_profile_train is None:
    df_img_profile_train = _compute_img_profile(df_train_all, "train_all")
    df_img_profile_train.to_parquet(img_profile_train_path, index=False)
    print("Saved:", img_profile_train_path)

if df_img_profile_test is None:
    df_img_profile_test = _compute_img_profile(df_test[df_test["has_file"]].copy(), "test")
    df_img_profile_test.to_parquet(img_profile_test_path, index=False)
    print("Saved:", img_profile_test_path)

# merge quick image stats into df_train_all (untuk ROI_LOOKUP di STAGE 2)
df_train_all = df_train_all.merge(
    df_img_profile_train[["case_id","variant","img_H","img_W","aspect","is_gray","bg_white_frac",
                          "roi_x0","roi_y0","roi_x1","roi_y1","roi_area_frac","edge_density"]],
    on=["case_id","variant"],
    how="left"
)

# build df_test_profile (buat TEST_ROI_LOOKUP di STAGE 2)
df_test_profile = df_img_profile_test.copy()
# merge kembali supaya df_test juga punya kolom profiling (opsional)
df_test = df_test.merge(
    df_test_profile[["case_id","variant","img_H","img_W","aspect","is_gray","bg_white_frac",
                     "roi_x0","roi_y0","roi_x1","roi_y1","roi_area_frac","edge_density"]],
    on=["case_id","variant"],
    how="left"
)

# ----------------------------
# 10) Mask profiling — cached + invalidate by SIG_ID
# ----------------------------
mask_profile_path = ART_DIR / "mask_profile.parquet"
need_mask_cols = {"case_id","variant","mask_H","mask_W","area_frac","n_components","mask_vs_image_mismatch","mask_outside_roi_frac"}

df_mask_profile = None
if _meta_matches() and mask_profile_path.exists():
    try:
        tmp = pd.read_parquet(mask_profile_path)
        if _df_cache_ok(tmp.rename(columns={"mask_H":"img_H","mask_W":"img_W"}), need_mask_cols):
            df_mask_profile = tmp
            print("\nLoaded cached mask_profile:", mask_profile_path, "| rows:", len(df_mask_profile))
    except Exception:
        df_mask_profile = None
else:
    if mask_profile_path.exists():
        print("\nmask_profile cache exists but meta mismatch -> recompute.")

if df_mask_profile is None:
    try:
        from tqdm.auto import tqdm
    except Exception:
        tqdm = lambda x, **kw: x

    print("\nComputing MASK profile on rows with has_mask=1 ...")
    df_forg2 = df_train_all[(df_train_all["has_mask"]==1)].copy()

    prof_rows = []
    blank_cnt = 0
    mismatch_between_masks_cnt = 0
    mismatch_with_image_cnt = 0
    read_fail_cnt = 0
    outside_roi_fail_cnt = 0

    for r in tqdm(df_forg2.itertuples(index=False), total=len(df_forg2)):
        cid = int(getattr(r, "case_id"))
        variant = str(getattr(r, "variant", ""))
        src = str(getattr(r, "source", ""))

        imgH = int(getattr(r, "img_H", -1))
        imgW = int(getattr(r, "img_W", -1))

        roi_x0 = int(getattr(r, "roi_x0", 0)); roi_y0 = int(getattr(r, "roi_y0", 0))
        roi_x1 = int(getattr(r, "roi_x1", 0)); roi_y1 = int(getattr(r, "roi_y1", 0))

        mpaths = getattr(r, "mask_paths")
        if not isinstance(mpaths, (list, tuple)) or len(mpaths) == 0:
            continue

        m_union = None
        raw_shapes = []
        used_transpose_to_match = False

        try:
            for mp in mpaths:
                raw = load_mask_raw(mp)
                m = to_2d_binary_mask(raw)
                raw_shapes.append(tuple(getattr(raw, "shape", ())))
                if m_union is None:
                    m_union = m
                else:
                    if m.shape != m_union.shape:
                        mismatch_between_masks_cnt += 1
                        # try transpose match
                        if m.T.shape == m_union.shape:
                            m = m.T
                        else:
                            # skip mask yang tidak bisa disejajarkan
                            continue
                    m_union = (m_union | m).astype(np.uint8)
        except Exception:
            read_fail_cnt += 1
            continue

        if m_union is None or m_union.ndim != 2:
            read_fail_cnt += 1
            continue

        Hm, Wm = m_union.shape
        pos = int(m_union.sum())
        area_frac = float(pos / max(Hm * Wm, 1))
        ncomp = count_components(m_union)
        if pos == 0:
            blank_cnt += 1

        # mismatch vs image (allow transpose match)
        img_mismatch = 0
        m_for_roi = m_union
        if imgH > 0 and imgW > 0:
            if (Hm == imgH and Wm == imgW):
                img_mismatch = 0
            elif (Wm == imgH and Hm == imgW) and (m_union.T.shape == (imgH, imgW)):
                img_mismatch = 0
                m_for_roi = m_union.T
                used_transpose_to_match = True
            else:
                img_mismatch = 1
                mismatch_with_image_cnt += 1

        # mask outside ROI fraction (only if mask aligned)
        mask_outside_roi_frac = np.nan
        if img_mismatch == 0 and imgH > 0 and imgW > 0:
            try:
                # clamp ROI
                x0 = max(0, min(roi_x0, imgW)); x1 = max(0, min(roi_x1, imgW))
                y0 = max(0, min(roi_y0, imgH)); y1 = max(0, min(roi_y1, imgH))
                if x1 > x0 and y1 > y0:
                    inside = m_for_roi[y0:y1, x0:x1]
                    inside_pos = int(inside.sum())
                    total_pos = int(m_for_roi.sum())
                    outside_pos = total_pos - inside_pos
                    mask_outside_roi_frac = float(outside_pos / total_pos) if total_pos > 0 else 0.0
                else:
                    mask_outside_roi_frac = 0.0
            except Exception:
                outside_roi_fail_cnt += 1
                mask_outside_roi_frac = np.nan

        prof_rows.append({
            "case_id": cid,
            "variant": variant,
            "source": src,
            "n_masks": int(getattr(r, "n_masks", len(mpaths))),
            "img_H": int(imgH),
            "img_W": int(imgW),
            "mask_H": int(Hm),
            "mask_W": int(Wm),
            "pos_pixels": int(pos),
            "area_frac": float(area_frac),
            "n_components": int(ncomp),
            "raw_shapes_unique": int(len(set(raw_shapes))),
            "mask_vs_image_mismatch": int(img_mismatch),
            "used_transpose_to_match_image": int(used_transpose_to_match),
            "mask_outside_roi_frac": float(mask_outside_roi_frac) if mask_outside_roi_frac == mask_outside_roi_frac else np.nan,
        })

    df_mask_profile = pd.DataFrame(prof_rows)
    df_mask_profile.to_parquet(mask_profile_path, index=False)

    print("\nSaved mask profile:", mask_profile_path, "| rows:", len(df_mask_profile))
    print("Read failures:", read_fail_cnt,
          "| blank masks:", blank_cnt,
          "| mask-shape mismatches (between masks):", mismatch_between_masks_cnt,
          "| mismatch with image:", mismatch_with_image_cnt,
          "| outside ROI calc fails:", outside_roi_fail_cnt)

# merge GT-derived columns into df_train_all (jelas dengan prefix gt_)
df_train_all = df_train_all.merge(
    df_mask_profile[["case_id","variant","area_frac","n_components","mask_vs_image_mismatch","mask_outside_roi_frac"]],
    on=["case_id","variant"],
    how="left"
)
df_train_all = df_train_all.rename(columns={
    "area_frac": "gt_area_frac",
    "n_components": "gt_n_components",
    "mask_vs_image_mismatch": "gt_mask_vs_image_mismatch",
    "mask_outside_roi_frac": "gt_mask_outside_roi_frac",
})

# ----------------------------
# 11) Case summary per case_id
# ----------------------------
def _has_variant(df, v):
    return int((df["variant"].astype(str) == v).any())

g = df_train_all.groupby("case_id", as_index=False).agg(
    has_auth=("variant", lambda x: int((pd.Series(x).astype(str)=="train_authentic").any())),
    has_forg=("variant", lambda x: int((pd.Series(x).astype(str)=="train_forged").any())),
    has_supp=("source", lambda x: int((pd.Series(x).astype(str)=="supplemental").any())),
    max_n_masks=("n_masks", lambda x: int(pd.to_numeric(pd.Series(x), errors="coerce").fillna(0).max())),
    n_rows=("case_id","size"),
)
df_case_summary = g.sort_values("case_id").reset_index(drop=True)
case_summary_path = ART_DIR / "case_summary.parquet"
df_case_summary.to_parquet(case_summary_path, index=False)

# ----------------------------
# 12) Summary + save reports + write meta
# ----------------------------
profile_summary = {
    "sig_id": SIG_ID,
    "train_rows": int(len(df_train_all)),
    "train_authentic_rows": int((df_train_all["y_forged"]==0).sum()),
    "train_forged_rows": int((df_train_all["y_forged"]==1).sum()),
    "train_forged_missing_masks": int(((df_train_all["y_forged"]==1) & (df_train_all["has_mask"]==0)).sum()),
    "supp_rows": int((df_train_all["source"]=="supplemental").sum()),
    "test_rows": int(len(df_test)),
    "test_missing_files_by_caseid_map": int((~df_test["has_file"]).sum()),
    "case_ids_total": int(df_case_summary["case_id"].nunique()),
    "case_ids_with_auth": int(df_case_summary["has_auth"].sum()),
    "case_ids_with_forg": int(df_case_summary["has_forg"].sum()),
    "case_ids_with_supp": int(df_case_summary["has_supp"].sum()),
}

# image profile summary (train)
v = df_img_profile_train.copy()
v = v[(v["img_H"] > 0) & (v["img_W"] > 0)]
if len(v):
    profile_summary["train_img_is_gray_rate"] = float(pd.to_numeric(v["is_gray"], errors="coerce").dropna().mean())
    profile_summary["train_img_bg_white_frac_mean"] = float(pd.to_numeric(v["bg_white_frac"], errors="coerce").dropna().mean())
    profile_summary["train_img_roi_area_frac_mean"] = float(pd.to_numeric(v["roi_area_frac"], errors="coerce").dropna().mean())
    profile_summary["train_img_edge_density_mean"] = float(pd.to_numeric(v["edge_density"], errors="coerce").dropna().mean())

# image profile summary (test)
vt = df_img_profile_test.copy()
vt = vt[(vt["img_H"] > 0) & (vt["img_W"] > 0)]
if len(vt):
    profile_summary["test_img_is_gray_rate"] = float(pd.to_numeric(vt["is_gray"], errors="coerce").dropna().mean())
    profile_summary["test_img_bg_white_frac_mean"] = float(pd.to_numeric(vt["bg_white_frac"], errors="coerce").dropna().mean())
    profile_summary["test_img_roi_area_frac_mean"] = float(pd.to_numeric(vt["roi_area_frac"], errors="coerce").dropna().mean())
    profile_summary["test_img_edge_density_mean"] = float(pd.to_numeric(vt["edge_density"], errors="coerce").dropna().mean())

# mask profile summary
if len(df_mask_profile):
    d = df_mask_profile["area_frac"].describe(percentiles=[0.01,0.05,0.1,0.5,0.9,0.95,0.99]).to_dict()
    profile_summary["mask_area_frac_desc"] = {k: float(v) for k,v in d.items() if pd.notna(v)}
    profile_summary["multi_mask_rate"] = float((pd.to_numeric(df_mask_profile["n_masks"], errors="coerce").fillna(0).astype(int) >= 2).mean())
    profile_summary["multi_component_rate"] = float((pd.to_numeric(df_mask_profile["n_components"], errors="coerce").fillna(0).astype(int) >= 2).mean())
    profile_summary["tiny_rate_area_lt_0p001"] = float((pd.to_numeric(df_mask_profile["area_frac"], errors="coerce").fillna(0) < 0.001).mean())
    profile_summary["huge_rate_area_gt_0p25"] = float((pd.to_numeric(df_mask_profile["area_frac"], errors="coerce").fillna(0) > 0.25).mean())
    profile_summary["mask_vs_image_mismatch_rate"] = float((pd.to_numeric(df_mask_profile["mask_vs_image_mismatch"], errors="coerce").fillna(0) == 1).mean())
    z = pd.to_numeric(df_mask_profile["mask_outside_roi_frac"], errors="coerce").dropna()
    if len(z):
        profile_summary["mask_outside_roi_frac_mean"] = float(z.mean())
else:
    profile_summary["mask_area_frac_desc"] = {}

# Persist outputs for next stages (lebih aman: simpan mask_paths sebagai JSON untuk parquet)
df_train_save = df_train_all.copy()
df_train_save["mask_paths_json"] = df_train_save["mask_paths"].apply(lambda x: json.dumps(list(x)) if isinstance(x, (list, tuple)) else "[]")
# drop list agar parquet lebih robust
df_train_save = df_train_save.drop(columns=["mask_paths"])

df_test_save = df_test.copy()

df_train_save.to_parquet(ART_DIR / "df_train_all.parquet", index=False)
df_test_save.to_parquet(ART_DIR / "df_test.parquet", index=False)
df_test_profile.to_parquet(ART_DIR / "df_test_profile.parquet", index=False)

with open(ART_DIR / "dataset_profile_summary.json", "w", encoding="utf-8") as f:
    json.dump(profile_summary, f, indent=2)

with open(ART_DIR / "paths.json", "w", encoding="utf-8") as f:
    json.dump({k: str(v) for k,v in PATHS.items()}, f, indent=2)

_write_meta(extra={"profile_summary_path": str(ART_DIR / "dataset_profile_summary.json")})

print("\n=== DATASET PROFILE SUMMARY ===")
for k, v in profile_summary.items():
    if k == "mask_area_frac_desc":
        print(f"- {k}:")
        for kk in ["count","mean","std","min","1%","5%","10%","50%","90%","95%","99%","max"]:
            if kk in v:
                print(f"    {kk:>4s}: {v[kk]:.6f}")
    else:
        print(f"- {k}: {v}")

print("\nSaved artifacts:")
print(" -", ART_DIR / "df_train_all.parquet", "(mask_paths disimpan sebagai mask_paths_json)")
print(" -", ART_DIR / "df_test.parquet")
print(" -", ART_DIR / "df_test_profile.parquet")
print(" -", img_profile_train_path)
print(" -", img_profile_test_path)
print(" -", mask_profile_path)
print(" -", case_summary_path)
print(" -", ART_DIR / "dataset_profile_summary.json")
print(" -", ART_DIR / "paths.json")
print(" -", META_PATH)

# ----------------------------
# 13) QUICK MASK FORMAT CHECK (3 nonblank + 3 blank if available)
#     handle duplicate case_id by picking best representative row per case_id
# ----------------------------
print("\n=== MASK FORMAT CHECK (3 nonblank + 3 blank if available) ===")

tmp = df_train_all.copy()
tmp["source_rank"] = tmp["source"].astype(str).map({"train": 0, "supplemental": 1}).fillna(2).astype(int)
tmp["n_masks_int"] = pd.to_numeric(tmp["n_masks"], errors="coerce").fillna(0).astype(int)
tmp = tmp.sort_values(["case_id", "source_rank", "n_masks_int"], ascending=[True, True, False])
tmp_u = tmp.drop_duplicates("case_id", keep="first")

lookup = tmp_u.set_index("case_id")[["image_path","mask_paths","variant","img_H","img_W"]].to_dict(orient="index")

def show_case(cid: int):
    info = lookup.get(int(cid), None)
    if info is None:
        print("case_id", cid, ": not found in lookup")
        return
    img_path = info["image_path"]
    mpaths = info["mask_paths"]
    variant = info.get("variant","")
    Himg = int(info.get("img_H", -1))
    Wimg = int(info.get("img_W", -1))

    if not mpaths:
        print(f"\ncase_id={cid} variant={variant} | no mask_paths")
        return

    mp0 = mpaths[0]
    raw = load_mask_raw(mp0)
    fixed = to_2d_binary_mask(raw)

    print(f"\ncase_id={cid} variant={variant}")
    print(" image shape:", (Himg, Wimg))
    print(" mask RAW  : dtype=", getattr(raw, "dtype", None), "ndim=", getattr(raw, "ndim", None), "shape=", getattr(raw, "shape", None))
    print(" mask FIXED:", "dtype=", fixed.dtype, "ndim=", fixed.ndim, "shape=", fixed.shape,
          "min/max=", int(fixed.min()) if fixed.size else None, int(fixed.max()) if fixed.size else None)

    if Himg > 0 and Wimg > 0:
        if fixed.shape == (Himg, Wimg) or fixed.T.shape == (Himg, Wimg):
            print(" OK: fixed mask matches image (direct or transpose).")
        else:
            print(" WARNING: fixed mask still not matching image size.")

if len(df_mask_profile):
    nonblank = df_mask_profile.sort_values("area_frac", ascending=False).head(3)["case_id"].tolist()
    blank = df_mask_profile[pd.to_numeric(df_mask_profile["area_frac"], errors="coerce").fillna(0) <= 0.0].head(3)["case_id"].tolist()
    for cid in nonblank:
        show_case(int(cid))
    for cid in blank:
        show_case(int(cid))

# ----------------------------
# 14) Export globals (keep context for next cells)
# ----------------------------
globals().update({
    "COMP_ROOT": COMP_ROOT,
    "PATHS": PATHS,
    "ART_DIR": ART_DIR,

    "df_train_all": df_train_all,                 # in-memory (mask_paths = list)
    "df_test": df_test,                           # in-memory (punya profiling kolom jika merge sukses)
    "df_test_profile": df_test_profile,           # penting untuk ROI lookup di STAGE 2
    "df_img_profile_train": df_img_profile_train,
    "df_img_profile_test": df_img_profile_test,
    "df_mask_profile": df_mask_profile,
    "df_case_summary": df_case_summary,
    "profile_summary": profile_summary,

    "DINO_MODEL_DIR": DINO_MODEL_DIR,
    "DINO_DIR": DINO_DIR,
    "SIG_ID": SIG_ID,
})

print("\nDONE. Exported objects:")
print("- PATHS, ART_DIR, df_train_all, df_test, df_test_profile")
print("- df_img_profile_train/test, df_mask_profile, df_case_summary, profile_summary")
print("- DINO_MODEL_DIR, DINO_DIR, SIG_ID")


DINO_MODEL_DIR: /kaggle/input/m/keras/dinov2/keras/dinov2_giant/1 | exists: True
DINO_DIR      : /kaggle/input/dinov2/pytorch/giant/1 | exists: True

COMP_ROOT: /kaggle/input/recodai-luc-scientific-image-forgery-detection

PATHS check:
 - TRAIN_AUTH  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images/authentic | exists=True
 - TRAIN_FORG  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images/forged | exists=True
 - TEST_IMAGES : /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images | exists=True
 - TRAIN_MASKS : /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_masks | exists=True
 - SUPP_IMAGES : /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_images | exists=True
 - SUPP_MASKS  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_masks | exists=True
 - SAMPLE_SUB  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv | exis

  0%|          | 0/2799 [00:00<?, ?it/s]


Saved mask profile: /kaggle/working/recodai_luc/artifacts/mask_profile.parquet | rows: 2799
Read failures: 0 | blank masks: 0 | mask-shape mismatches (between masks): 0 | mismatch with image: 0 | outside ROI calc fails: 0

=== DATASET PROFILE SUMMARY ===
- sig_id: 93e24d45b9
- train_rows: 5176
- train_authentic_rows: 2377
- train_forged_rows: 2799
- train_forged_missing_masks: 0
- supp_rows: 48
- test_rows: 1
- test_missing_files_by_caseid_map: 0
- case_ids_total: 2795
- case_ids_with_auth: 2377
- case_ids_with_forg: 2751
- case_ids_with_supp: 48
- train_img_is_gray_rate: 0.49748840803709427
- train_img_bg_white_frac_mean: 0.009860023518081532
- train_img_roi_area_frac_mean: 0.9999183272051363
- train_img_edge_density_mean: 0.10090139136446119
- test_img_is_gray_rate: 1.0
- test_img_bg_white_frac_mean: 0.6939644191576086
- test_img_roi_area_frac_mean: 0.9704340532606417
- test_img_edge_density_mean: 0.1612283457880435
- mask_area_frac_desc:
    count: 2799.000000
    mean: 0.055985
  

# Data, Labels, CV & Sanity Guards

In [ ]:
# ============================================================
# STAGE 1 — Data, Labels, Image+Mask Profiling, CV & Sanity Guards (ONE CELL)
# REVISI FULL v3.4 (TEST-STUB SAFE)
#
# Upgrade utama (sesuai kasus kamu: sample_submission + test_images cuma 1 data):
# - sample_submission parser lebih fleksibel (id/case_id + annotation/prediction/rle/kolom lain)
# - Deteksi "TEST STUB" otomatis:
#     * jika df_test kecil (<=5) ATAU jumlah file test_images < jumlah baris submission
#   maka missing test file dianggap NORMAL (INFO, bukan WARN)
# - Profiling test dibuat opsional & auto-disable saat TEST_STUB untuk hemat waktu + tidak misleading
# - tqdm pakai "tqdm" standar (tanpa "Loading widget...")
#
# Export globals (dipakai stage berikutnya):
# - DATA_ROOT, PATHS, RUN_DIR, ART_DIR, DINO_BASE_DIR
# - df_train_all (sample-level), df_train_seg (mask-only + nonblank), df_train_cls (classification)
# - df_pairs (case_id pairs auth+forg), df_test (aligned to sample_submission)
# - df_img_profile, df_test_profile, df_mask_profile, df_case_summary
# - fold_map, N_FOLDS, SEED
# - rle_encode, rle_decode, union_masks_aligned, load_mask_any, to_binary_mask
# ============================================================

import os, re, json, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------
# Helpers: safe prints
# ----------------------------
def _p(s):
    print(s, flush=True)

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **kw: x

SEED = int(globals().get("SEED", 2025))
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# 0) Auto-detect DATA_ROOT
# ============================================================
DEFAULT_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")

def auto_find_data_root(default_root: Path) -> Path:
    if default_root.exists():
        return default_root
    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("Tidak menemukan /kaggle/input. Pastikan run di Kaggle + Add Input kompetisi.")
    cands = []
    for d in base.iterdir():
        if not d.is_dir():
            continue
        ok = (d / "sample_submission.csv").exists() and (d / "train_images").exists() and (d / "test_images").exists()
        if ok:
            cands.append(d)
    if not cands:
        for f in base.rglob("sample_submission.csv"):
            root = f.parent
            if (root / "train_images").exists() and (root / "test_images").exists():
                cands.append(root)
    if not cands:
        raise FileNotFoundError(
            f"DATA_ROOT default tidak ada: {default_root}\n"
            "Tidak ada kandidat root dengan (sample_submission.csv, train_images, test_images) di /kaggle/input."
        )
    def score(d: Path) -> int:
        s = 0
        for need in ["train_masks", "supplemental_images", "supplemental_masks"]:
            if (d / need).exists():
                s += 1
        name = d.name.lower()
        if ("recodai" in name) or ("luc" in name) or ("forgery" in name):
            s += 2
        return s
    cands = sorted(cands, key=score, reverse=True)
    if len(cands) > 1:
        _p("[WARN] Banyak kandidat DATA_ROOT. Dipilih yang paling lengkap:")
        for c in cands[:6]:
            _p(f"  - {c} (score={score(c)})")
    return cands[0]

DATA_ROOT = auto_find_data_root(DEFAULT_ROOT)

# ============================================================
# 1) PATHS + RUN/ART dirs
# ============================================================
PATHS = {
    "ROOT":         DATA_ROOT,
    "TRAIN_IMAGES": DATA_ROOT / "train_images",
    "TRAIN_MASKS":  DATA_ROOT / "train_masks",
    "TEST_IMAGES":  DATA_ROOT / "test_images",
    "SAMPLE_SUB":   DATA_ROOT / "sample_submission.csv",
    "SUPP_IMAGES":  DATA_ROOT / "supplemental_images",
    "SUPP_MASKS":   DATA_ROOT / "supplemental_masks",
    "TRAIN_AUTH":   (DATA_ROOT / "train_images" / "authentic"),
    "TRAIN_FORG":   (DATA_ROOT / "train_images" / "forged"),
}

RUN_DIR = Path("/kaggle/working/recodai_luc")
ART_DIR = RUN_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

# DINO path (prioritaskan model kamu)
DINO_PREF = Path("/kaggle/input/m/keras/dinov2/keras/dinov2_giant/1")
DINO_BASE_DIR = Path(globals().get("DINO_BASE_DIR", str(DINO_PREF)))
if (not DINO_BASE_DIR.exists()) and DINO_PREF.exists():
    DINO_BASE_DIR = DINO_PREF

# Validasi minimal
for k in ["TRAIN_IMAGES","TRAIN_MASKS","TEST_IMAGES","SAMPLE_SUB"]:
    if not PATHS[k].exists():
        raise FileNotFoundError(f"PATH tidak ditemukan: {k} -> {PATHS[k]}")

_p("OK PATHS:")
_p(f"  data_root : {DATA_ROOT}")
for k in ["TRAIN_AUTH","TRAIN_FORG","TRAIN_MASKS","SUPP_IMAGES","SUPP_MASKS","TEST_IMAGES","SAMPLE_SUB"]:
    _p(f"  {k:10s}: {PATHS[k]} | exists={PATHS[k].exists()}")
_p(f"  RUN_DIR   : {RUN_DIR}")
_p(f"  ART_DIR   : {ART_DIR}")
_p(f"  DINO_BASE : {DINO_BASE_DIR} | exists={DINO_BASE_DIR.exists()}")

with open(ART_DIR / "paths_stage1.json", "w", encoding="utf-8") as f:
    json.dump({k: str(v) for k, v in PATHS.items()} | {"RUN_DIR": str(RUN_DIR), "ART_DIR": str(ART_DIR), "DINO_BASE_DIR": str(DINO_BASE_DIR)}, f, indent=2)

# ============================================================
# 2) Indexing helpers
# ============================================================
IMG_EXTS  = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
MASK_EXTS = {".npy",".npz"} | IMG_EXTS

def folder_stats(folder: Path, allow_ext=None, show=6):
    if not folder.exists():
        _p(f"\n[{folder}] MISSING")
        return []
    files = [p for p in folder.rglob("*") if p.is_file()]
    if allow_ext is not None:
        files = [p for p in files if p.suffix.lower() in allow_ext]
    exts = Counter([p.suffix.lower() for p in files])
    _p(f"\n[{folder}] files={len(files):,} ext_counts={dict(exts.most_common(8))}")
    for p in files[:show]:
        try:
            _p("  - " + str(p.relative_to(folder)))
        except Exception:
            _p("  - " + str(p))
    return files

def extract_case_id(stem: str) -> str:
    s = str(stem)
    m = re.search(r"(\d+)", s)
    if m:
        return str(int(m.group(1)))  # normalize "00045" -> "45"
    return s.strip()

def list_images(folder: Path):
    if not folder.exists():
        return []
    out = []
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            cid = extract_case_id(p.stem)
            out.append((cid, p))
    out.sort(key=lambda x: (int(x[0]) if x[0].isdigit() else x[0], str(x[1])))
    return out

def list_masks(folder: Path):
    if not folder.exists():
        return []
    out = []
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in MASK_EXTS:
            cid = extract_case_id(p.stem)
            out.append((cid, p))
    out.sort(key=lambda x: (int(x[0]) if x[0].isdigit() else x[0], str(x[1])))
    return out

def group_files_by_case(pairs):
    mp = defaultdict(list)
    for cid, p in pairs:
        mp[str(cid)].append(p)
    for cid in mp:
        mp[cid] = sorted(mp[cid], key=lambda x: (x.suffix.lower(), x.name))
    return dict(mp)

# stats (anti salah root)
_ = folder_stats(PATHS["TRAIN_IMAGES"], allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["TRAIN_MASKS"],  allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["SUPP_IMAGES"],  allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["SUPP_MASKS"],   allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["TEST_IMAGES"],  allow_ext=IMG_EXTS)

# ============================================================
# 3) Build image maps (auth/forg/supp/test)
# ============================================================
train_auth = {cid: p for cid, p in list_images(PATHS["TRAIN_AUTH"])} if PATHS["TRAIN_AUTH"].exists() else {}
train_forg = {cid: p for cid, p in list_images(PATHS["TRAIN_FORG"])} if PATHS["TRAIN_FORG"].exists() else {}

if len(train_auth) == 0 and len(train_forg) == 0:
    _p("[WARN] Tidak menemukan subfolder authentic/forged. Fallback scan train_images path string.")
    for cid, p in list_images(PATHS["TRAIN_IMAGES"]):
        s = str(p).lower()
        if "authentic" in s:
            train_auth[cid] = p
        elif "forged" in s:
            train_forg[cid] = p

supp_map = {cid: p for cid, p in list_images(PATHS["SUPP_IMAGES"])} if PATHS["SUPP_IMAGES"].exists() else {}
test_map = {cid: p for cid, p in list_images(PATHS["TEST_IMAGES"])}

overlap = set(train_auth) & set(train_forg)
_p("\nIndexed images:")
_p(f"  train/authentic: {len(train_auth):,}")
_p(f"  train/forged   : {len(train_forg):,}")
_p(f"  overlap case_id : {len(overlap):,} (normal: pasangan auth+forg)")
_p(f"  supplemental   : {len(supp_map):,}")
_p(f"  test files     : {len(test_map):,}")

train_mask_map = group_files_by_case(list_masks(PATHS["TRAIN_MASKS"]))
supp_mask_map  = group_files_by_case(list_masks(PATHS["SUPP_MASKS"]))

_p("\nGrouped masks:")
_p(f"  train_masks groups: {len(train_mask_map):,}")
_p(f"  supp_masks  groups: {len(supp_mask_map):,}")

def get_mask_list(cid: str, prefer_train=True):
    cid = str(cid)
    lst = []
    if prefer_train and cid in train_mask_map:
        lst += train_mask_map[cid]
    if (not prefer_train) and cid in supp_mask_map:
        lst += supp_mask_map[cid]
    if prefer_train and cid in supp_mask_map:
        lst += supp_mask_map[cid]
    if (not prefer_train) and cid in train_mask_map:
        lst += train_mask_map[cid]
    seen, out = set(), []
    for p in lst:
        sp = str(p)
        if sp not in seen:
            out.append(p)
            seen.add(sp)
    return out

# ============================================================
# 4) Mask loader + binarize + align (robust)
# ============================================================
def load_mask_any(path: str) -> np.ndarray:
    p = Path(path)
    ext = p.suffix.lower()
    if ext in IMG_EXTS:
        with Image.open(p) as im:
            return np.array(im.convert("L"))
    if ext == ".npy":
        return np.array(np.load(p, allow_pickle=False))
    if ext == ".npz":
        z = np.load(p, allow_pickle=False)
        key = "arr_0" if "arr_0" in z.files else (z.files[0] if len(z.files) else None)
        if key is None:
            raise ValueError(f"NPZ kosong: {p}")
        return np.array(z[key])
    raise ValueError(f"Mask ext tidak didukung: {ext} ({p})")

def to_binary_mask(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 0:
        return np.zeros((0, 0), dtype=np.uint8)

    # normalize to 2D
    while a.ndim > 2:
        if a.ndim == 3:
            if (a.shape[0] <= 64) and (a.shape[1] > 16) and (a.shape[2] > 16):
                a = a.max(axis=0); continue
            if (a.shape[2] <= 16) and (a.shape[0] > 16) and (a.shape[1] > 16):
                a = a.max(axis=2); continue
            a = np.squeeze(a)
            if a.ndim == 3:
                k = int(np.argmin(list(a.shape)))
                a = a.max(axis=k)
            continue
        if a.ndim == 4:
            a = np.squeeze(a)
            if a.ndim == 4:
                k = int(np.argmin(list(a.shape)))
                a = a.max(axis=k)
            continue
        a = np.squeeze(a)
        if a.ndim > 2:
            a = a.reshape(a.shape[-2], a.shape[-1])

    if a.ndim != 2:
        a = np.zeros((0, 0), dtype=np.uint8)

    if np.issubdtype(a.dtype, np.floating):
        return (a > 0.5).astype(np.uint8)
    return (a > 0).astype(np.uint8)

def align_mask_to_image(mask2d: np.ndarray, H: int, W: int) -> np.ndarray:
    m = np.asarray(mask2d)
    if m.ndim == 1 and m.size == H * W:
        try:
            m = m.reshape(H, W)
        except Exception:
            pass

    if m.shape == (H, W):
        return m.astype(np.uint8)
    if m.shape == (W, H):
        return m.T.astype(np.uint8)

    if m.size == H * W:
        try:
            return m.reshape(H, W).astype(np.uint8)
        except Exception:
            pass

    if m.size == 0:
        return np.zeros((H, W), dtype=np.uint8)

    im = Image.fromarray((m > 0).astype(np.uint8) * 255)
    im = im.resize((W, H), resample=Image.NEAREST)
    return (np.array(im) > 0).astype(np.uint8)

def union_masks_aligned(mask_paths: list, H: int, W: int) -> np.ndarray:
    if not mask_paths:
        return None
    out = np.zeros((H, W), dtype=np.uint8)
    for p in mask_paths:
        raw = load_mask_any(p)
        bm  = to_binary_mask(raw)
        am  = align_mask_to_image(bm, H, W)
        out = np.maximum(out, am)
    return out

# ============================================================
# 5) Build df_train_all (sample-level unique) + df_pairs
# ============================================================
rows = []

for cid, p in train_auth.items():
    rows.append({
        "sample_id": f"{cid}__auth",
        "case_id": str(cid),
        "variant": "auth",
        "source": "train",
        "image_path": str(p),
        "mask_paths": [],
        "n_masks": 0,
        "y_forged": 0,
    })

missing_train_masks = []
for cid, p in train_forg.items():
    mlist = get_mask_list(str(cid), prefer_train=True)
    if len(mlist) == 0:
        missing_train_masks.append(str(cid))
    rows.append({
        "sample_id": f"{cid}__forg",
        "case_id": str(cid),
        "variant": "forg",
        "source": "train",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": 1,
    })

missing_supp_masks = []
for cid, p in supp_map.items():
    mlist = get_mask_list(str(cid), prefer_train=False)
    if len(mlist) == 0:
        missing_supp_masks.append(str(cid))
        yv = np.nan
    else:
        yv = 1
    rows.append({
        "sample_id": f"{cid}__supp",
        "case_id": str(cid),
        "variant": "supp",
        "source": "supplemental",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": yv,
    })

df_train_all = pd.DataFrame(rows).sort_values(["case_id","variant"]).reset_index(drop=True)

df_pairs = (df_train_all.pivot_table(
    index="case_id",
    columns="variant",
    values="image_path",
    aggfunc="first"
).reset_index())
if "auth" in df_pairs.columns and "forg" in df_pairs.columns:
    df_pairs = df_pairs.dropna(subset=["auth","forg"]).copy()
else:
    df_pairs = df_pairs.iloc[0:0].copy()

forg_mask = df_train_all[df_train_all["variant"].isin(["forg","supp"])].groupby("case_id")["mask_paths"].first()
df_pairs["mask_paths"] = df_pairs["case_id"].map(lambda x: forg_mask.get(str(x), []))
df_pairs["n_masks"] = df_pairs["mask_paths"].map(lambda x: len(x) if isinstance(x, list) else 0)

_p("\nTrain build summary:")
_p(f"  df_train_all rows: {len(df_train_all):,}")
_p("  variant counts:")
_p(df_train_all["variant"].value_counts().to_string())
_p("\n  y_forged counts (including NaN):")
_p(df_train_all["y_forged"].value_counts(dropna=False).to_string())

if missing_train_masks:
    _p(f"\n[WARN] forged train tanpa mask: {len(missing_train_masks):,} | contoh: {', '.join(missing_train_masks[:20])}")
if missing_supp_masks:
    _p(f"\n[WARN] supplemental tanpa mask: {len(missing_supp_masks):,} | contoh: {', '.join(missing_supp_masks[:20])}")

df_train_cls = df_train_all[df_train_all["y_forged"].notna()].copy()
df_train_cls["y_forged"] = df_train_cls["y_forged"].astype(int)

# ============================================================
# 6) df_test aligned to sample_submission (FLEXIBLE)
# ============================================================
df_sub = pd.read_csv(PATHS["SAMPLE_SUB"])
cols_lower = {c.lower(): c for c in df_sub.columns}

# id col
id_col = None
for cand in ["case_id","id","image_id","object_id"]:
    if cand in cols_lower:
        id_col = cols_lower[cand]
        break
if id_col is None:
    id_col = df_sub.columns[0]  # fallback: first col

# annotation/pred col (optional)
ann_col = None
for cand in ["annotation","prediction","rle","encodedpixels","encoded_pixels","mask","label","target"]:
    if cand in cols_lower:
        ann_col = cols_lower[cand]
        break
if ann_col is None:
    # fallback: first non-id col, if exists
    non_id = [c for c in df_sub.columns if c != id_col]
    ann_col = non_id[0] if non_id else None

df_sub = df_sub.rename(columns={id_col: "case_id"}).copy()
df_sub["case_id"] = df_sub["case_id"].map(lambda x: extract_case_id(str(x))).astype(str)

df_test = df_sub[["case_id"]].copy()
df_test["image_path"] = df_test["case_id"].map(lambda cid: str(test_map.get(str(cid), "")))
df_test["has_file"] = df_test["image_path"].map(lambda p: Path(p).exists())

resolved = int(df_test["has_file"].sum())
n_test = int(len(df_test))

TEST_STUB = bool((n_test <= 5) or (len(test_map) < n_test))
_p(f"\nTest aligned: resolved {resolved:,}/{n_test:,} | test_files={len(test_map):,} | TEST_STUB={TEST_STUB}")

if resolved != n_test:
    bad = df_test.loc[~df_test["has_file"], "case_id"].tolist()
    if TEST_STUB:
        _p("[INFO] Ada case_id test yang tidak resolve ke file. Ini NORMAL untuk public/stub test.")
    else:
        _p("[WARN] Ada case_id test yang tidak resolve ke file. Cek dataset root / penamaan file.")
    _p("  contoh missing: " + ", ".join(bad[:20]))

# ============================================================
# 7) RLE utils (order='F')
# ============================================================
RLE_ORDER = "F"

def rle_decode(rle: str, shape_hw: tuple, order: str="F") -> np.ndarray:
    H, W = shape_hw
    s = str(rle).strip()
    if s == "" or s.lower() == "authentic":
        return np.zeros((H, W), dtype=np.uint8)
    nums = np.asarray(list(map(int, s.split())), dtype=np.int64)
    if len(nums) % 2 != 0:
        raise ValueError("Invalid RLE (odd length)")
    starts = nums[0::2] - 1
    lens   = nums[1::2]
    ends = starts + lens
    flat = np.zeros(H*W, dtype=np.uint8)
    for st, en in zip(starts, ends):
        st = max(0, int(st)); en = min(int(en), flat.size)
        if en > st:
            flat[st:en] = 1
    if order.upper() == "F":
        return flat.reshape((W, H)).T
    return flat.reshape((H, W))

def rle_encode(mask: np.ndarray, order: str="F") -> str:
    m = (np.asarray(mask) > 0).astype(np.uint8)
    if m.sum() == 0:
        return ""
    pixels = m.T.reshape(-1) if order.upper() == "F" else m.reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# ============================================================
# 8) IMAGE PROFILING (cached)
# ============================================================
img_profile_path  = ART_DIR / "image_profile.parquet"
test_profile_path = ART_DIR / "test_image_profile.parquet"

def _img_cache_broken(df: pd.DataFrame) -> bool:
    if df is None or len(df) == 0:
        return False
    need = {"sample_id","case_id","variant","img_H","img_W","bg_white_frac","roi_area_frac","edge_density","is_gray"}
    if not need.issubset(set(df.columns)):
        return True
    try:
        if float(df["img_H"].median()) < 32 or float(df["img_W"].median()) < 32:
            return True
    except Exception:
        return True
    return False

def compute_image_quick_stats(
    img_path: str,
    max_side: int = 512,
    bg_thr: int = 250,
    min_valid_frac: float = 0.05,
    margin_px: int = 8,
    edge_thr: float = 20.0
):
    with Image.open(img_path) as im:
        im = im.convert("RGB")
        W, H = im.size
        im_small = im.copy()
        im_small.thumbnail((max_side, max_side), resample=Image.BILINEAR)
        arr = np.array(im_small, dtype=np.uint8)

    hs, ws = arr.shape[:2]
    g = arr.mean(axis=2).astype(np.float32)

    rg = np.abs(arr[...,0].astype(np.int16) - arr[...,1].astype(np.int16))
    gb = np.abs(arr[...,1].astype(np.int16) - arr[...,2].astype(np.int16))
    is_gray = float((rg.mean() + gb.mean()) / 2.0 < 1.5)

    bg_white_frac = float((g >= bg_thr).mean())

    valid = (g < bg_thr)
    ys, xs = np.where(valid)
    if len(xs) < int(min_valid_frac * hs * ws):
        x0s, y0s, x1s, y1s = 0, 0, ws, hs
    else:
        x0s, x1s = int(xs.min()), int(xs.max())
        y0s, y1s = int(ys.min()), int(ys.max())
        msx = max(2, int(margin_px * ws / max(W, 1)))
        msy = max(2, int(margin_px * hs / max(H, 1)))
        x0s = max(0, x0s - msx); y0s = max(0, y0s - msy)
        x1s = min(ws, x1s + msx); y1s = min(hs, y1s + msy)

    x0 = int(round(x0s * W / max(ws, 1)))
    x1 = int(round(x1s * W / max(ws, 1)))
    y0 = int(round(y0s * H / max(hs, 1)))
    y1 = int(round(y1s * H / max(hs, 1)))
    x0 = max(0, min(x0, W)); x1 = max(0, min(x1, W))
    y0 = max(0, min(y0, H)); y1 = max(0, min(y1, H))
    if (x1 <= x0 + 4) or (y1 <= y0 + 4):
        x0, y0, x1, y1 = 0, 0, W, H

    roi_area_frac = float(((x1 - x0) * (y1 - y0)) / max(W * H, 1))

    dx = np.abs(np.diff(g, axis=1))
    dy = np.abs(np.diff(g, axis=0))
    dxp = np.pad(dx, ((0,0),(0,1)), mode="edge")
    dyp = np.pad(dy, ((0,1),(0,0)), mode="edge")
    edge_density = float(((dxp + dyp) > edge_thr).mean())

    return {
        "img_H": int(H),
        "img_W": int(W),
        "aspect": float(W / max(H, 1)),
        "is_gray": float(is_gray),
        "bg_white_frac": float(bg_white_frac),
        "roi_x0": int(x0), "roi_y0": int(y0), "roi_x1": int(x1), "roi_y1": int(y1),
        "roi_area_frac": float(roi_area_frac),
        "edge_density": float(edge_density),
    }

# train image profile
df_img_profile = None
if img_profile_path.exists():
    try:
        tmp = pd.read_parquet(img_profile_path)
        if _img_cache_broken(tmp):
            _p("\nFound cached image_profile but looks broken. Recomputing...")
        else:
            df_img_profile = tmp
            _p(f"\nLoaded cached image profile: {img_profile_path} | rows={len(df_img_profile):,}")
    except Exception:
        df_img_profile = None

if df_img_profile is None:
    _p("\nComputing IMAGE profile on df_train_all ...")
    rows_img = []
    fail = 0
    for r in tqdm(df_train_all.itertuples(index=False), total=len(df_train_all)):
        sid = getattr(r, "sample_id")
        cid = getattr(r, "case_id")
        var = getattr(r, "variant")
        src = getattr(r, "source")
        ip  = getattr(r, "image_path")
        try:
            st = compute_image_quick_stats(ip)
            st.update({"sample_id": str(sid), "case_id": str(cid), "variant": str(var), "source": str(src), "image_path": str(ip)})
            rows_img.append(st)
        except Exception:
            fail += 1
            rows_img.append({
                "sample_id": str(sid), "case_id": str(cid), "variant": str(var), "source": str(src), "image_path": str(ip),
                "img_H": -1, "img_W": -1, "aspect": np.nan, "is_gray": np.nan,
                "bg_white_frac": np.nan, "roi_x0": 0, "roi_y0": 0, "roi_x1": 0, "roi_y1": 0,
                "roi_area_frac": np.nan, "edge_density": np.nan
            })
    df_img_profile = pd.DataFrame(rows_img)
    df_img_profile.to_parquet(img_profile_path, index=False)
    _p(f"Saved: {img_profile_path} | rows={len(df_img_profile):,} | fails={fail}")

df_train_all = df_train_all.merge(
    df_img_profile[["sample_id","img_H","img_W","aspect","is_gray","bg_white_frac","roi_x0","roi_y0","roi_x1","roi_y1","roi_area_frac","edge_density"]],
    on="sample_id", how="left"
)

# test profile (AUTO: disable by default on TEST_STUB)
PROFILE_TEST = int(globals().get("PROFILE_TEST", 0 if TEST_STUB else 1))
df_test_profile = None

if PROFILE_TEST == 1 and test_profile_path.exists():
    try:
        df_test_profile = pd.read_parquet(test_profile_path)
        _p(f"\nLoaded cached test image profile: {test_profile_path} | rows={len(df_test_profile):,}")
    except Exception:
        df_test_profile = None

if df_test_profile is None:
    if PROFILE_TEST == 1:
        _p("\nComputing TEST image profile ...")
        trows = []
        fail = 0
        for r in tqdm(df_test.itertuples(index=False), total=len(df_test)):
            cid = getattr(r, "case_id")
            ip  = getattr(r, "image_path")
            hasf = bool(getattr(r, "has_file"))
            if (not hasf) or (not Path(ip).exists()):
                trows.append({
                    "case_id": str(cid), "image_path": str(ip), "has_file": bool(hasf),
                    "img_H": -1, "img_W": -1, "roi_x0": 0, "roi_y0": 0, "roi_x1": 0, "roi_y1": 0,
                    "bg_white_frac": np.nan, "roi_area_frac": np.nan, "edge_density": np.nan, "is_gray": np.nan, "aspect": np.nan
                })
                continue
            try:
                st = compute_image_quick_stats(ip)
                st.update({"case_id": str(cid), "image_path": str(ip), "has_file": True})
                trows.append(st)
            except Exception:
                fail += 1
                trows.append({
                    "case_id": str(cid), "image_path": str(ip), "has_file": True,
                    "img_H": -1, "img_W": -1, "roi_x0": 0, "roi_y0": 0, "roi_x1": 0, "roi_y1": 0,
                    "bg_white_frac": np.nan, "roi_area_frac": np.nan, "edge_density": np.nan, "is_gray": np.nan, "aspect": np.nan
                })
        df_test_profile = pd.DataFrame(trows)
        df_test_profile.to_parquet(test_profile_path, index=False)
        _p(f"Saved: {test_profile_path} | rows={len(df_test_profile):,} | fails={fail}")
    else:
        _p("\n[INFO] Skip TEST image profiling (PROFILE_TEST=0). Ini normal saat TEST_STUB.")
        df_test_profile = df_test[["case_id","image_path","has_file"]].copy()
        # tetap simpan minimal agar stage berikutnya tidak bingung
        df_test_profile.to_parquet(test_profile_path, index=False)
        _p(f"Saved minimal: {test_profile_path} | rows={len(df_test_profile):,}")

# ============================================================
# 9) MASK PROFILING (cached) + mask_outside_roi_frac
# ============================================================
mask_profile_path = ART_DIR / "mask_profile.parquet"

_CC = "none"
try:
    import cv2
    _CC = "cv2"
except Exception:
    try:
        import scipy.ndimage as ndi
        _CC = "scipy"
    except Exception:
        _CC = "none"

def count_components(bin_mask: np.ndarray) -> int:
    if bin_mask is None or bin_mask.size == 0:
        return 0
    m = (bin_mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return 0
    if _CC == "cv2":
        n, _ = cv2.connectedComponents(m, connectivity=8)
        return int(max(n - 1, 0))
    if _CC == "scipy":
        _, n = ndi.label(m.astype(bool))
        return int(n)
    return -1

def _mask_cache_broken(df: pd.DataFrame) -> bool:
    if df is None or len(df) == 0:
        return False
    need = {"sample_id","case_id","variant","img_H","img_W","pos_pixels","area_frac","n_components","mask_vs_image_mismatch","mask_outside_roi_frac"}
    if not need.issubset(set(df.columns)):
        return True
    return False

def compute_mask_profile(df_samples: pd.DataFrame) -> pd.DataFrame:
    prof = []
    read_fail = 0
    blank = 0
    mismatch = 0
    outside_roi_fail = 0

    for r in tqdm(df_samples.itertuples(index=False), total=len(df_samples)):
        sid = getattr(r, "sample_id")
        cid = getattr(r, "case_id")
        var = getattr(r, "variant")
        src = getattr(r, "source")
        ip  = getattr(r, "image_path")
        mps = getattr(r, "mask_paths")

        if not isinstance(mps, (list, tuple)) or len(mps) == 0:
            continue
        if not Path(ip).exists():
            continue

        H = int(getattr(r, "img_H", -1))
        W = int(getattr(r, "img_W", -1))
        if H <= 0 or W <= 0:
            try:
                with Image.open(ip) as im:
                    W, H = im.size
            except Exception:
                read_fail += 1
                continue

        try:
            # union + track raw mismatch vs image
            out = np.zeros((H, W), dtype=np.uint8)
            raw_mismatch = 0
            for p in mps:
                raw = load_mask_any(p)
                bm  = to_binary_mask(raw)
                # mismatch check BEFORE resize
                sh = tuple(getattr(bm, "shape", ()))
                ok_shape = (sh == (H, W)) or (sh == (W, H)) or (bm.size == H*W)
                if not ok_shape:
                    raw_mismatch = 1
                am  = align_mask_to_image(bm, H, W)
                out = np.maximum(out, am)

            pos = int(out.sum())
            if pos == 0:
                blank += 1
            af = float(pos) / float(max(H * W, 1))
            ncomp = count_components(out)
            if raw_mismatch:
                mismatch += 1

            roi_x0 = int(getattr(r, "roi_x0", 0))
            roi_y0 = int(getattr(r, "roi_y0", 0))
            roi_x1 = int(getattr(r, "roi_x1", W))
            roi_y1 = int(getattr(r, "roi_y1", H))
            outside_frac = np.nan
            try:
                if roi_x1 > roi_x0 and roi_y1 > roi_y0 and pos > 0:
                    inside = out[roi_y0:roi_y1, roi_x0:roi_x1]
                    inside_pos = int(inside.sum())
                    outside_pos = pos - inside_pos
                    outside_frac = float(outside_pos / max(pos, 1))
                else:
                    outside_frac = 0.0
            except Exception:
                outside_roi_fail += 1
                outside_frac = np.nan

            prof.append({
                "sample_id": str(sid),
                "case_id": str(cid),
                "variant": str(var),
                "source": str(src),
                "img_H": int(H),
                "img_W": int(W),
                "pos_pixels": int(pos),
                "area_frac": float(af),
                "n_components": int(ncomp),
                "mask_vs_image_mismatch": int(raw_mismatch),
                "mask_outside_roi_frac": float(outside_frac) if outside_frac == outside_frac else np.nan,
            })
        except Exception:
            read_fail += 1
            continue

    dfp = pd.DataFrame(prof)
    _p(f"\n[mask_profile] computed rows={len(dfp):,} | read_fail={read_fail} | blank={blank} | mismatch={mismatch} | outside_roi_fail={outside_roi_fail} | cc_backend={_CC}")
    return dfp

df_has_masks = df_train_all[pd.to_numeric(df_train_all["n_masks"], errors="coerce").fillna(0).astype(int) > 0].copy()

df_mask_profile = None
if mask_profile_path.exists():
    try:
        tmp = pd.read_parquet(mask_profile_path)
        if _mask_cache_broken(tmp):
            _p("\nFound cached mask_profile but looks broken/outdated. Recomputing...")
        else:
            df_mask_profile = tmp
            _p(f"\nLoaded cached mask profile: {mask_profile_path} | rows={len(df_mask_profile):,}")
    except Exception:
        df_mask_profile = None

if df_mask_profile is None:
    df_mask_profile = compute_mask_profile(df_has_masks)
    df_mask_profile.to_parquet(mask_profile_path, index=False)
    _p(f"Saved mask profile: {mask_profile_path}")

df_train_all = df_train_all.merge(
    df_mask_profile[["sample_id","pos_pixels","area_frac","n_components","mask_vs_image_mismatch","mask_outside_roi_frac"]],
    on="sample_id", how="left"
).rename(columns={
    "pos_pixels":"gt_pos_pixels",
    "area_frac":"gt_area_frac",
    "n_components":"gt_n_components",
    "mask_vs_image_mismatch":"gt_mask_vs_image_mismatch",
    "mask_outside_roi_frac":"gt_mask_outside_roi_frac",
})

df_train_seg = df_train_all[
    (pd.to_numeric(df_train_all["n_masks"], errors="coerce").fillna(0).astype(int) > 0) &
    (pd.to_numeric(df_train_all["gt_pos_pixels"], errors="coerce").fillna(0).astype(int) > 0)
].copy().reset_index(drop=True)

_p(f"\nSeg train (mask-only, nonblank): {len(df_train_seg):,} / mask_samples={len(df_has_masks):,}")

# ============================================================
# 10) Sanity checks (mask format + RLE roundtrip)
# ============================================================
def quick_mask_format_check(df_profile, k_nonblank=3, k_blank=3):
    if df_profile is None or len(df_profile) == 0:
        _p("\n[Sanity] mask_profile kosong (tidak ada mask yang terbaca).")
        return
    nb = df_profile[df_profile["pos_pixels"] > 0].head(k_nonblank)
    bl = df_profile[df_profile["pos_pixels"] == 0].head(k_blank)
    samp = pd.concat([nb, bl], axis=0)

    _p("\n=== MASK FORMAT CHECK (nonblank + blank if available) ===")
    lookup = df_train_all.set_index("sample_id")[["image_path","mask_paths","variant","case_id","img_H","img_W"]].to_dict(orient="index")

    for _, rr in samp.iterrows():
        sid = rr["sample_id"]
        meta = lookup.get(sid, None)
        if meta is None:
            continue
        ip = meta["image_path"]
        mps = meta["mask_paths"]
        cid = meta["case_id"]
        var = meta["variant"]
        H = int(meta.get("img_H", -1))
        W = int(meta.get("img_W", -1))
        if (H <= 0 or W <= 0) and Path(ip).exists():
            with Image.open(ip) as im:
                W, H = im.size

        if not Path(ip).exists() or not mps:
            continue

        raw = load_mask_any(mps[0])
        bm  = to_binary_mask(raw)
        am  = align_mask_to_image(bm, H, W)

        _p(f"\ncase_id={cid} variant={var} sample_id={sid}")
        _p(f" image shape: (H,W)=({H},{W})")
        _p(f" mask RAW  : dtype={getattr(raw,'dtype',None)} ndim={getattr(raw,'ndim',None)} shape={getattr(raw,'shape',None)}")
        _p(f" mask FIXED: dtype={am.dtype} ndim={am.ndim} shape={am.shape} min/max={int(am.min())}/{int(am.max())}")
        if am.shape == (H, W):
            _p(" OK: fixed mask matches image.")
        else:
            _p(" WARN: fixed mask mismatch (cek loader/align).")

quick_mask_format_check(df_mask_profile, 3, 3)

rt_n = min(20, len(df_train_seg))
rt_ok = 0
rt_seen = 0
shape_mis = 0
area_fracs = []

if rt_n > 0:
    samp = df_train_seg.sample(rt_n, random_state=SEED)
    for r in samp.itertuples(index=False):
        ip = getattr(r, "image_path")
        mps = getattr(r, "mask_paths")
        H = int(getattr(r, "img_H", -1))
        W = int(getattr(r, "img_W", -1))
        if (H <= 0 or W <= 0) and Path(ip).exists():
            with Image.open(ip) as im:
                W, H = im.size
        if not Path(ip).exists() or not mps:
            continue
        um = union_masks_aligned(mps, H, W)
        if um is None:
            continue
        if um.shape != (H, W):
            shape_mis += 1
        s = int(um.sum())
        if s > 0:
            area_fracs.append(s / float(H*W))
        enc = rle_encode(um, order=RLE_ORDER)
        dec = rle_decode(enc, (H, W), order=RLE_ORDER)
        rt_seen += 1
        if np.array_equal(um, dec):
            rt_ok += 1

_p("\nSanity (RLE roundtrip on seg samples):")
_p(f"  sampled={rt_n} | tested={rt_seen} | ok={rt_ok} | shape_mismatch={shape_mis} | RLE_ORDER='{RLE_ORDER}'")
if area_fracs:
    _p(f"  area_frac min/med/max: {min(area_fracs):.6f}/{float(np.median(area_fracs)):.6f}/{max(area_fracs):.6f}")

# ============================================================
# 11) CV split (group by case_id) — ROBUST (no crash strata kecil)
# ============================================================
def _nanmax_safe(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(x.max()) if len(x) else 0.0

grp = df_train_all.groupby("case_id").agg(
    has_auth=("variant", lambda x: int("auth" in set(x))),
    has_forg=("variant", lambda x: int(("forg" in set(x)) or ("supp" in set(x)))),
    source_bin=("source", lambda x: "supp" if ("supplemental" in set(x)) else "train"),
    max_area=("gt_area_frac", _nanmax_safe),
    max_comp=("gt_n_components", _nanmax_safe),
    n_samples=("sample_id", "count"),
).reset_index()

valid_area = grp["max_area"].to_numpy()
valid_area = valid_area[np.isfinite(valid_area)]
q = np.quantile(valid_area, [0.2, 0.4, 0.6, 0.8]).tolist() if len(valid_area) else [0.0, 0.0, 0.0, 0.0]

def area_bin(a: float):
    if not np.isfinite(a) or a <= 0:
        return "A0"
    if a <= q[0]: return "A1"
    if a <= q[1]: return "A2"
    if a <= q[2]: return "A3"
    if a <= q[3]: return "A4"
    return "A5"

def comp_bin(c: float):
    if not np.isfinite(c) or c <= 0:
        return "C0"
    if c <= 1: return "C1"
    if c <= 2: return "C2"
    if c <= 4: return "C3"
    return "C4"

grp["area_bin"] = grp["max_area"].map(area_bin)
grp["comp_bin"] = grp["max_comp"].map(comp_bin)

def _make_key(df: pd.DataFrame, cols: list) -> pd.Series:
    if not cols:
        return pd.Series(["ALL"] * len(df), index=df.index)
    parts = [df[c].astype(str) for c in cols]
    key = parts[0]
    for p in parts[1:]:
        key = key + "_" + p
    return key

def _collapse_rare(key: pd.Series, min_count: int) -> pd.Series:
    vc = key.value_counts()
    rare = set(vc[vc < int(min_count)].index.tolist())
    if not rare:
        return key
    return key.where(~key.isin(rare), other="RARE")

def assign_group_folds(grp_df: pd.DataFrame, desired_folds: int, seed: int):
    n_cases = len(grp_df)
    desired_folds = int(min(max(2, desired_folds), n_cases))
    if n_cases < 2:
        raise ValueError(f"case_id terlalu sedikit untuk CV: n_cases={n_cases}")

    strat_levels = [
        ("H+S+Area+Comp", ["has_auth","source_bin","area_bin","comp_bin"]),
        ("H+S+Area",      ["has_auth","source_bin","area_bin"]),
        ("H+S",           ["has_auth","source_bin"]),
        ("H+Area",        ["has_auth","area_bin"]),
        ("H",             ["has_auth"]),
        ("S",             ["source_bin"]),
        ("NONE",          []),
    ]

    chosen = None
    chosen_key = None
    chosen_k = None
    chosen_mode = None  # "stratified" or "kfold"

    for name, cols in strat_levels:
        raw_key = _make_key(grp_df, cols)
        for k in range(desired_folds, 1, -1):
            key = _collapse_rare(raw_key, min_count=k)
            vc = key.value_counts()
            if int(vc.shape[0]) < 2:
                continue
            if int(vc.min()) >= k:
                chosen = (name, cols)
                chosen_key = key
                chosen_k = k
                chosen_mode = "stratified"
                break
        if chosen_mode == "stratified":
            break

    grp_out = grp_df.copy()
    grp_out["fold"] = -1

    if chosen_mode == "stratified":
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=int(chosen_k), shuffle=True, random_state=int(seed))
        y = chosen_key.astype(str).values
        for f, (_, va) in enumerate(skf.split(np.zeros(len(grp_out)), y)):
            grp_out.loc[grp_out.index[va], "fold"] = int(f)
        info = {
            "mode": "StratifiedKFold",
            "n_splits": int(chosen_k),
            "strat_level": chosen[0],
            "n_classes": int(chosen_key.nunique()),
            "min_class_count": int(chosen_key.value_counts().min()),
        }
        return grp_out, info

    from sklearn.model_selection import KFold
    k = int(min(desired_folds, n_cases))
    if k < 2:
        k = 2
    kf = KFold(n_splits=k, shuffle=True, random_state=int(seed))
    for f, (_, va) in enumerate(kf.split(np.zeros(n_cases))):
        grp_out.loc[grp_out.index[va], "fold"] = int(f)

    info = {
        "mode": "KFold(group-only, shuffled)",
        "n_splits": int(k),
        "strat_level": "NONE",
        "n_classes": 0,
        "min_class_count": 0,
    }
    return grp_out, info

N_FOLDS_REQ = int(globals().get("N_FOLDS", 5))
grp, cv_info = assign_group_folds(grp, desired_folds=N_FOLDS_REQ, seed=SEED)
N_FOLDS = int(grp["fold"].nunique())

fold_map = dict(zip(grp["case_id"].astype(str), grp["fold"].astype(int)))
df_train_all["fold"] = df_train_all["case_id"].map(fold_map).astype(int)
df_train_seg["fold"] = df_train_seg["case_id"].map(fold_map).astype(int)
df_train_cls["fold"] = df_train_cls["case_id"].map(fold_map).astype(int)

_p(f"\nCV ready (group case_id): n_splits={N_FOLDS} seed={SEED}")
_p("CV mode:")
_p("  " + json.dumps(cv_info, indent=2))
_p("\nfold counts (case_id):")
_p(grp["fold"].value_counts().sort_index().to_string())

# ============================================================
# 12) Case summary + save artifacts
# ============================================================
df_case_summary = df_train_all.groupby("case_id", as_index=False).agg(
    has_auth=("variant", lambda x: int("auth" in set(x))),
    has_forg=("variant", lambda x: int(("forg" in set(x)) or ("supp" in set(x)))),
    has_supp=("source", lambda x: int("supplemental" in set(x))),
    max_n_masks=("n_masks", lambda x: int(pd.to_numeric(pd.Series(x), errors="coerce").fillna(0).max())),
    max_area=("gt_area_frac", _nanmax_safe),
    max_comp=("gt_n_components", _nanmax_safe),
    n_rows=("sample_id","count"),
).sort_values("case_id").reset_index(drop=True)

# ============================================================
# 13) Save artifacts
# ============================================================
df_train_all.to_parquet(ART_DIR / "df_train_all.parquet", index=False)
df_train_seg.to_parquet(ART_DIR / "df_train_seg.parquet", index=False)
df_train_cls.to_parquet(ART_DIR / "df_train_cls.parquet", index=False)
df_pairs.to_parquet(ART_DIR / "df_pairs.parquet", index=False)
df_test.to_parquet(ART_DIR / "df_test.parquet", index=False)

df_img_profile.to_parquet(img_profile_path, index=False)
df_test_profile.to_parquet(test_profile_path, index=False)
df_mask_profile.to_parquet(mask_profile_path, index=False)

grp.to_csv(ART_DIR / "cv_case_folds.csv", index=False)
df_train_all[["sample_id","case_id","variant","source","fold"]].to_csv(ART_DIR / "cv_sample_folds.csv", index=False)
df_case_summary.to_parquet(ART_DIR / "case_summary.parquet", index=False)

stage1_summary = {
    "seed": SEED,
    "n_folds": int(N_FOLDS),
    "cv_info": cv_info,
    "n_train_all": int(len(df_train_all)),
    "n_train_cls": int(len(df_train_cls)),
    "n_train_seg": int(len(df_train_seg)),
    "n_pairs": int(len(df_pairs)),
    "n_test": int(len(df_test)),
    "test_files": int(len(test_map)),
    "test_resolved": int(resolved),
    "test_stub": bool(TEST_STUB),
    "profile_test": int(PROFILE_TEST),
    "img_profile_rows": int(len(df_img_profile)),
    "test_profile_rows": int(len(df_test_profile)),
    "mask_profile_rows": int(len(df_mask_profile)),
    "cc_backend": _CC,
    "DINO_BASE_DIR": str(DINO_BASE_DIR),
}
with open(ART_DIR / "stage1_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage1_summary, f, indent=2)

_p("\nSaved artifacts:")
for fp in [
    ART_DIR/"df_train_all.parquet",
    ART_DIR/"df_train_seg.parquet",
    ART_DIR/"df_train_cls.parquet",
    ART_DIR/"df_pairs.parquet",
    ART_DIR/"df_test.parquet",
    img_profile_path,
    test_profile_path,
    mask_profile_path,
    ART_DIR/"case_summary.parquet",
    ART_DIR/"cv_case_folds.csv",
    ART_DIR/"cv_sample_folds.csv",
    ART_DIR/"stage1_summary.json",
]:
    _p(f"  - {fp}")

_p("\nHeads:")
_p("\ndf_train_all.head():\n" + df_train_all.head(5).to_string(index=False))
_p("\ndf_train_seg.head():\n" + df_train_seg.head(5).to_string(index=False))
_p("\ndf_test.head():\n" + df_test.head(5).to_string(index=False))

_p("\nDONE. Exported objects in globals:")
_p("- DATA_ROOT, PATHS, RUN_DIR, ART_DIR, DINO_BASE_DIR")
_p("- df_train_all, df_train_seg, df_train_cls, df_pairs, df_test")
_p("- df_img_profile, df_test_profile, df_mask_profile, df_case_summary")
_p("- fold_map, N_FOLDS, SEED")
_p("- RLE_ORDER, rle_encode, rle_decode")
_p("- load_mask_any, to_binary_mask, union_masks_aligned")

globals().update({
    "DATA_ROOT": DATA_ROOT,
    "PATHS": PATHS,
    "RUN_DIR": RUN_DIR,
    "ART_DIR": ART_DIR,
    "DINO_BASE_DIR": DINO_BASE_DIR,
    "df_train_all": df_train_all,
    "df_train_seg": df_train_seg,
    "df_train_cls": df_train_cls,
    "df_pairs": df_pairs,
    "df_test": df_test,
    "df_img_profile": df_img_profile,
    "df_test_profile": df_test_profile,
    "df_mask_profile": df_mask_profile,
    "df_case_summary": df_case_summary,
    "fold_map": fold_map,
    "N_FOLDS": int(N_FOLDS),
    "SEED": SEED,
    "RLE_ORDER": RLE_ORDER,
    "rle_encode": rle_encode,
    "rle_decode": rle_decode,
    "load_mask_any": load_mask_any,
    "to_binary_mask": to_binary_mask,
    "union_masks_aligned": union_masks_aligned,
    "TEST_STUB": bool(TEST_STUB),
    "PROFILE_TEST": int(PROFILE_TEST),
})


OK PATHS:
  data_root : /kaggle/input/recodai-luc-scientific-image-forgery-detection
  TRAIN_AUTH: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images/authentic | exists=True
  TRAIN_FORG: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images/forged | exists=True
  TRAIN_MASKS: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_masks | exists=True
  SUPP_IMAGES: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_images | exists=True
  SUPP_MASKS: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_masks | exists=True
  TEST_IMAGES: /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images | exists=True
  SAMPLE_SUB: /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv | exists=True
  RUN_DIR   : /kaggle/working/recodai_luc
  ART_DIR   : /kaggle/working/recodai_luc/artifacts
  DINO_BASE : /kaggle/input/m/keras/dinov2/keras/dinov2_giant/1 | e

 44%|████▎     | 1219/2799 [00:10<00:12, 129.35it/s]

# DINOv2 Feature Cache (CPU-Optimized)

In [ ]:
# ============================================================
# STAGE 2 — DINOv2-LARGE Feature Cache (ONE CELL) — REVISI FULL v4.0
# FORCE PATH: /kaggle/input/dinov2/pytorch/large/1
#
# Output per item (kompatibel STAGE 3):
# - patch_desc: (H_p, W_p, D_fused) float16 (L2-normalized)
# - cls_desc  : (D_fused,) float16 (L2-normalized)
# - meta      : JSON string
#
# NO-SKIP:
# - missing file -> placeholder
# - fail extract -> placeholder
# - FORCE_REBUILD=1 (default) -> tidak reuse cache lama
# ============================================================

import os, json, math, gc, time, hashlib, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn.functional as F
from transformers import AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ----------------------------
# Guards
# ----------------------------
need_cols = {"sample_id","case_id","image_path","variant","y_forged","fold"}
if "df_train_all" not in globals() or not isinstance(df_train_all, pd.DataFrame):
    raise RuntimeError("df_train_all belum ada. Jalankan STAGE 1 dulu.")
if not need_cols.issubset(set(df_train_all.columns)):
    raise RuntimeError(f"df_train_all missing cols {need_cols - set(df_train_all.columns)}")

if "df_test" not in globals() or not isinstance(df_test, pd.DataFrame):
    raise RuntimeError("df_test belum ada. Jalankan STAGE 1 dulu.")
if not {"case_id","image_path"}.issubset(set(df_test.columns)):
    raise RuntimeError("df_test harus punya kolom case_id, image_path.")

# ----------------------------
# FORCE DINOv2 LARGE path (sesuai request)
# ----------------------------
DINO_LARGE_DIR = Path("/kaggle/input/dinov2/pytorch/large/1")
if not DINO_LARGE_DIR.exists():
    raise FileNotFoundError(f"DINO_LARGE_DIR tidak ditemukan: {DINO_LARGE_DIR}")

SEED = int(globals().get("SEED", 2025))
random.seed(SEED)
np.random.seed(SEED)

# ----------------------------
# Config (samakan gaya setting yang kamu minta)
# ----------------------------
PATCH_SIZE = 14

USE_MULTI_SCALE = True
MAX_SIDE_BASE   = 384
MAX_SIDE_HI     = 512
MIN_SIDE        = 224

FUSE_MODE = "concat"          # "concat" atau "avg"
USE_LIGHT_WHITEN = True
WHITEN_EPS = 1e-6

USE_FP16_STORE = True
N_LIMIT_TRAIN  = None
N_LIMIT_TEST   = None

# NO-SKIP behavior
FORCE_REBUILD = int(globals().get("FORCE_REBUILD", 1))              # default: 1 (rebuild semua)
PLACEHOLDER_ON_MISSING = int(globals().get("PLACEHOLDER_ON_MISSING", 1))  # default: 1
PLACEHOLDER_ON_FAIL    = int(globals().get("PLACEHOLDER_ON_FAIL", 1))     # default: 1

# threads (CPU) konservatif + auto GPU kalau ada
try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
    torch.set_num_interop_threads(1)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(device.type == "cuda")
try:
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

CFG = {
    "seed": SEED,
    "patch_size": PATCH_SIZE,
    "use_multi_scale": USE_MULTI_SCALE,
    "max_side_base": MAX_SIDE_BASE,
    "max_side_hi": (MAX_SIDE_HI if USE_MULTI_SCALE else MAX_SIDE_BASE),
    "min_side": MIN_SIDE,
    "fuse_mode": FUSE_MODE,
    "use_light_whiten": USE_LIGHT_WHITEN,
    "whiten_eps": WHITEN_EPS,
    "use_fp16_store": USE_FP16_STORE,
    "dino_large_dir": str(DINO_LARGE_DIR),
    "force_rebuild": int(FORCE_REBUILD),
    "placeholder_on_missing": int(PLACEHOLDER_ON_MISSING),
    "placeholder_on_fail": int(PLACEHOLDER_ON_FAIL),
    "device": str(device),
    "amp": bool(USE_AMP),
}
CFG_ID = hashlib.sha1(json.dumps(CFG, sort_keys=True).encode("utf-8")).hexdigest()[:12]

CACHE_ROOT  = Path("/kaggle/working/recodai_luc/cache/dino_v2_large") / f"cfg_{CFG_ID}"
CACHE_TRAIN = CACHE_ROOT / "train_all"
CACHE_TEST  = CACHE_ROOT / "test"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_TRAIN.mkdir(parents=True, exist_ok=True)
CACHE_TEST.mkdir(parents=True, exist_ok=True)

with open(CACHE_ROOT / "cfg.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)

print("CFG_ID:", CFG_ID)
print("CACHE_ROOT:", CACHE_ROOT)
print("DINO_LARGE_DIR:", DINO_LARGE_DIR)
print("device:", device, "| AMP:", bool(USE_AMP))
print("FORCE_REBUILD:", FORCE_REBUILD, "| PH_MISSING:", PLACEHOLDER_ON_MISSING, "| PH_FAIL:", PLACEHOLDER_ON_FAIL)

# ----------------------------
# Normalization (ImageNet)
# ----------------------------
IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def _ceil_to_multiple(x, m):
    return int(max(m, math.ceil(float(x)/m) * m))

def resize_keep_ar(img: Image.Image, max_side: int, min_side: int, patch: int):
    W0, H0 = img.size
    long0 = max(W0, H0)
    if long0 <= 0:
        return img, (H0, W0), (H0, W0)

    target_long = min(max_side, max(min_side, long0))
    scale = target_long / float(long0)

    W1 = max(patch, int(round(W0 * scale)))
    H1 = max(patch, int(round(H0 * scale)))

    W1 = _ceil_to_multiple(W1, patch)
    H1 = _ceil_to_multiple(H1, patch)

    if (W1, H1) != (W0, H0):
        img = img.resize((W1, H1), resample=Image.BICUBIC)

    return img, (H0, W0), (H1, W1)

def pil_to_tensor_norm(img: Image.Image):
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    arr = (arr - IMNET_MEAN) / IMNET_STD
    return torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).contiguous()

def l2_normalize(x: torch.Tensor, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True) + eps)

def _infer_grid_from_tokens(N: int, H1: int, W1: int, patch: int):
    gh = max(1, H1 // patch)
    gw = max(1, W1 // patch)
    if gh * gw == N:
        return gh, gw
    ar = W1 / max(1, H1)
    gw0 = int(round(math.sqrt(N * ar)))
    gw0 = max(1, min(gw0, N))
    gh0 = max(1, N // gw0)
    gw0 = max(1, N // gh0)
    if gh0 * gw0 != N:
        return 1, N
    return gh0, gw0

# ----------------------------
# Load DINOv2 LARGE (local, robust)
# ----------------------------
print("\n[LOAD] DINOv2-LARGE from:", DINO_LARGE_DIR)
model = None
load_err = None
for kwargs in (
    {"local_files_only": True, "trust_remote_code": True},
    {"local_files_only": True},
):
    try:
        model = AutoModel.from_pretrained(str(DINO_LARGE_DIR), **kwargs)
        load_err = None
        break
    except Exception as e:
        load_err = e

if model is None:
    raise RuntimeError(f"Gagal load DINOv2-LARGE. Last err: {type(load_err).__name__}: {load_err}")

model.eval().to(device)

with torch.inference_mode():
    dummy = torch.zeros((1,3,224,224), dtype=torch.float32, device=device)
    out = model(pixel_values=dummy)
    D0 = int(out.last_hidden_state.shape[-1])
print("[OK] Model loaded. embed_dim:", D0)

@torch.inference_mode()
def _forward_dino(img_pil: Image.Image):
    x = pil_to_tensor_norm(img_pil).to(device, dtype=torch.float32)
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            y = model(pixel_values=x).last_hidden_state
    else:
        y = model(pixel_values=x).last_hidden_state
    cls = y[:, 0, :]
    pt  = y[:, 1:, :]
    return cls, pt

def _tokens_to_grid(pt: torch.Tensor, H1: int, W1: int, patch: int):
    N = int(pt.shape[1])
    D = int(pt.shape[2])
    gh, gw = _infer_grid_from_tokens(N, H1, W1, patch)
    pt0 = pt.squeeze(0)
    pt0 = l2_normalize(pt0, dim=-1)
    try:
        grid = pt0.reshape(gh, gw, D).contiguous()
    except Exception:
        grid = pt0.reshape(1, N, D).contiguous()
        gh, gw = 1, N
    return grid, gh, gw, D

def _resample_grid_to(grid: torch.Tensor, gh_t: int, gw_t: int):
    gh, gw, D = grid.shape
    x = grid.permute(2,0,1).unsqueeze(0).contiguous()
    x2 = F.interpolate(x, size=(gh_t, gw_t), mode="bilinear", align_corners=False)
    return x2.squeeze(0).permute(1,2,0).contiguous()

def _light_whiten_patches(grid: torch.Tensor, eps: float = 1e-6):
    gh, gw, D = grid.shape
    x = grid.reshape(-1, D)
    mu = x.mean(dim=0, keepdim=True)
    sd = x.std(dim=0, keepdim=True).clamp_min(eps)
    xw = (x - mu) / sd
    xw = l2_normalize(xw, dim=-1)
    return xw.reshape(gh, gw, D).contiguous()

# placeholder dims
D_FUSED = (2 * D0) if (USE_MULTI_SCALE and FUSE_MODE == "concat") else D0
GH_PH = max(1, _ceil_to_multiple(MIN_SIDE, PATCH_SIZE) // PATCH_SIZE)
GW_PH = GH_PH

def make_placeholder_feats(reason: str):
    patch = np.zeros((GH_PH, GW_PH, D_FUSED), dtype=np.float32)
    cls   = np.zeros((D_FUSED,), dtype=np.float32)
    patch[0,0,0] = 1.0
    cls[0] = 1.0
    patch = patch.reshape(-1, D_FUSED)
    patch = patch / (np.linalg.norm(patch, axis=-1, keepdims=True) + 1e-12)
    patch = patch.reshape(GH_PH, GW_PH, D_FUSED)
    cls = cls / (np.linalg.norm(cls) + 1e-12)
    meta = {
        "cfg_id": CFG_ID,
        "backend": "hf_torch_large",
        "patch_size": int(PATCH_SIZE),
        "grid_hw": [int(GH_PH), int(GW_PH)],
        "embed_dim_raw": int(D0),
        "embed_dim_patch": int(D_FUSED),
        "embed_dim_cls": int(D_FUSED),
        "use_multi_scale": bool(USE_MULTI_SCALE),
        "fuse_mode": str(FUSE_MODE),
        "use_light_whiten": False,
        "placeholder": True,
        "placeholder_reason": str(reason),
        "device": str(device),
        "amp": bool(USE_AMP),
    }
    return patch, cls, meta

def extract_features_one(image_path: str):
    with Image.open(image_path) as im:
        img0 = im.convert("RGB")

    # BASE
    img_b, (H0,W0), (Hb,Wb) = resize_keep_ar(img0, max_side=MAX_SIDE_BASE, min_side=MIN_SIDE, patch=PATCH_SIZE)
    cls_b, pt_b = _forward_dino(img_b)
    cls_b = l2_normalize(cls_b, dim=-1).squeeze(0)
    grid_b, gh, gw, D = _tokens_to_grid(pt_b, Hb, Wb, PATCH_SIZE)

    if USE_MULTI_SCALE:
        img_h, _, (Hh,Wh) = resize_keep_ar(img0, max_side=MAX_SIDE_HI, min_side=MIN_SIDE, patch=PATCH_SIZE)
        cls_h, pt_h = _forward_dino(img_h)
        cls_h = l2_normalize(cls_h, dim=-1).squeeze(0)
        grid_h, _, _, _ = _tokens_to_grid(pt_h, Hh, Wh, PATCH_SIZE)

        grid_h_rs = _resample_grid_to(grid_h, gh, gw)
        grid_h_rs = l2_normalize(grid_h_rs.reshape(-1, D), dim=-1).reshape(gh, gw, D).contiguous()

        if FUSE_MODE == "avg":
            grid_f = l2_normalize((grid_b + grid_h_rs), dim=-1)
            cls_f  = l2_normalize((cls_b + cls_h), dim=-1)
        else:
            grid_f = torch.cat([grid_b, grid_h_rs], dim=-1)
            grid_f = l2_normalize(grid_f.reshape(-1, grid_f.shape[-1]), dim=-1).reshape(gh, gw, -1).contiguous()
            cls_f  = torch.cat([cls_b, cls_h], dim=-1)
            cls_f  = l2_normalize(cls_f, dim=-1)
    else:
        grid_f = grid_b
        cls_f  = cls_b

    if USE_LIGHT_WHITEN:
        grid_f = _light_whiten_patches(grid_f, eps=WHITEN_EPS)

    meta = {
        "cfg_id": CFG_ID,
        "backend": "hf_torch_large",
        "orig_hw": [int(H0), int(W0)],
        "resized_hw_base": [int(Hb), int(Wb)],
        "patch_size": int(PATCH_SIZE),
        "grid_hw": [int(grid_f.shape[0]), int(grid_f.shape[1])],
        "embed_dim_raw": int(D0),
        "embed_dim_patch": int(grid_f.shape[-1]),
        "embed_dim_cls": int(cls_f.shape[0]),
        "max_side_base": int(MAX_SIDE_BASE),
        "max_side_hi": int(MAX_SIDE_HI) if USE_MULTI_SCALE else int(MAX_SIDE_BASE),
        "min_side": int(MIN_SIDE),
        "use_multi_scale": bool(USE_MULTI_SCALE),
        "fuse_mode": str(FUSE_MODE),
        "use_light_whiten": bool(USE_LIGHT_WHITEN),
        "placeholder": False,
        "device": str(device),
        "amp": bool(USE_AMP),
    }
    return grid_f, cls_f, meta

def save_npz(path: Path, patch_desc, cls_desc, meta: dict):
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(patch_desc, torch.Tensor):
        pd_np = patch_desc.detach().cpu().numpy()
    else:
        pd_np = np.asarray(patch_desc)

    if isinstance(cls_desc, torch.Tensor):
        cd_np = cls_desc.detach().cpu().numpy()
    else:
        cd_np = np.asarray(cls_desc)

    if USE_FP16_STORE:
        pd_np = pd_np.astype(np.float16)
        cd_np = cd_np.astype(np.float16)
    else:
        pd_np = pd_np.astype(np.float32)
        cd_np = cd_np.astype(np.float32)

    np.savez(str(path), patch_desc=pd_np, cls_desc=cd_np, meta=json.dumps(meta))

def cache_loop(df: pd.DataFrame, id_col: str, out_dir: Path, limit=None, label="train"):
    n = len(df) if limit is None else min(len(df), int(limit))
    ok = 0
    rebuilt = 0
    reused = 0
    ph = 0
    miss = 0
    fail = 0
    t0 = time.time()

    rows = df.head(n).itertuples(index=False)
    for i, r in enumerate(rows, start=1):
        uid = str(getattr(r, id_col))
        ip  = str(getattr(r, "image_path", ""))
        out = out_dir / f"{uid}.npz"

        if out.exists():
            if FORCE_REBUILD == 1:
                try:
                    out.unlink()
                except Exception:
                    pass
                rebuilt += 1
            else:
                reused += 1
                ok += 1
                continue

        has_file = bool(ip) and Path(ip).exists()
        if not has_file:
            miss += 1
            if PLACEHOLDER_ON_MISSING == 1:
                pt, cls, meta = make_placeholder_feats(reason="missing_image")
                save_npz(out, pt, cls, meta)
                ph += 1
                ok += 1
            continue

        try:
            pt, cls, meta = extract_features_one(ip)
            save_npz(out, pt, cls, meta)
            ok += 1
        except Exception as e:
            fail += 1
            if PLACEHOLDER_ON_FAIL == 1:
                pt, cls, meta = make_placeholder_feats(reason=f"extract_fail:{type(e).__name__}")
                save_npz(out, pt, cls, meta)
                ph += 1
                ok += 1
            else:
                print(f"[{label}] FAIL uid={uid} path={ip} err={type(e).__name__}: {e}")

        if (i % 50) == 0:
            dt = time.time() - t0
            rate = i / max(dt, 1e-9)
            print(f"[{label}] {i}/{n} | ok={ok} | reused={reused} | rebuilt={rebuilt} | ph={ph} | miss={miss} | fail={fail} | {rate:.2f} it/s | elapsed={dt:.1f}s")
            gc.collect()
            if device.type == "cuda":
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass

    dt = time.time() - t0
    print(f"\n[{label}] DONE. total={n} ok={ok} reused={reused} rebuilt={rebuilt} ph={ph} miss={miss} fail={fail} elapsed={dt:.1f}s")
    return {"label": label, "total": int(n), "cached_ok": int(ok), "reused": int(reused), "rebuilt": int(rebuilt),
            "placeholder": int(ph), "miss": int(miss), "fail": int(fail), "elapsed_s": float(dt)}

def build_manifest(df: pd.DataFrame, id_col: str, out_dir: Path):
    recs = []
    for r in df.itertuples(index=False):
        uid = str(getattr(r, id_col))
        case_id = str(getattr(r, "case_id", ""))
        variant = str(getattr(r, "variant", "")) if hasattr(r, "variant") else ""
        fold = int(getattr(r, "fold", -1)) if hasattr(r, "fold") else -1
        y_forged = getattr(r, "y_forged", -1)
        try:
            y_forged = int(y_forged)
        except Exception:
            y_forged = -1
        ip = str(getattr(r, "image_path", ""))

        has_file = int(bool(ip) and Path(ip).exists())
        p = out_dir / f"{uid}.npz"
        recs.append({
            id_col: uid,
            "case_id": case_id,
            "variant": variant,
            "fold": fold,
            "y_forged": y_forged,
            "image_path": ip,
            "has_file": has_file,
            "feat_path": str(p),
            "feat_exists": int(p.exists()),
        })
    return pd.DataFrame(recs)

# ----------------------------
# Run
# ----------------------------
df_train_run = df_train_all.reset_index(drop=True).copy()
df_test_run  = df_test.reset_index(drop=True).copy()

df_train_run = df_train_run.sort_values(["variant","fold","case_id","sample_id"]).reset_index(drop=True)
df_test_run  = df_test_run.sort_values(["case_id"]).reset_index(drop=True)

print("\nCaching TRAIN_ALL features...")
rep_train = cache_loop(df_train_run, id_col="sample_id", out_dir=CACHE_TRAIN, limit=N_LIMIT_TRAIN, label="train_all")

print("\nCaching TEST features...")
rep_test = cache_loop(df_test_run, id_col="case_id", out_dir=CACHE_TEST, limit=N_LIMIT_TEST, label="test")

man_train = build_manifest(df_train_run, "sample_id", CACHE_TRAIN)
man_test  = build_manifest(df_test_run,  "case_id",  CACHE_TEST)

MAN_TRAIN_PATH = CACHE_ROOT / "manifest_train_all.csv"
MAN_TEST_PATH  = CACHE_ROOT / "manifest_test.csv"
man_train.to_csv(MAN_TRAIN_PATH, index=False)
man_test.to_csv(MAN_TEST_PATH, index=False)

summary = {
    "cfg": CFG,
    "cfg_id": CFG_ID,
    "cache_root": str(CACHE_ROOT),
    "train": rep_train,
    "test": rep_test,
}
with open(CACHE_ROOT / "cache_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nWrote manifests:")
print(" -", MAN_TRAIN_PATH, "| feat_exists mean:", float(man_train["feat_exists"].mean()) if len(man_train) else 0.0)
print(" -", MAN_TEST_PATH,  "| feat_exists mean:", float(man_test["feat_exists"].mean()) if len(man_test) else 0.0)
print(" -", CACHE_ROOT / "cache_summary.json")

globals().update({
    "DINO_DIR": DINO_LARGE_DIR,
    "DINO_BACKEND": "hf_torch_large",
    "DINO_MODEL_TORCH": model,
    "CACHE_ROOT": CACHE_ROOT,
    "CACHE_TRAIN": CACHE_TRAIN,
    "CACHE_TEST": CACHE_TEST,
    "MAN_TRAIN_PATH": MAN_TRAIN_PATH,
    "MAN_TEST_PATH": MAN_TEST_PATH,
    "CFG_ID": CFG_ID,
    "DINO_CFG": CFG,
    "extract_features_one": extract_features_one,
})

print("\nDONE. Exported objects:")
print("- DINO_DIR, CACHE_ROOT, MAN_TRAIN_PATH, MAN_TEST_PATH, CFG_ID")
print("- helper: extract_features_one()")


# Robust Matching (Top-k + MNN + Multi-Peak Translation)

In [ ]:
# ============================================================
# STAGE 3 — Robust Matching (Top-k + TRUE MNN + Ratio/Margin + Peak NMS) (ONE CELL)
# REVISI FULL v6.3 (TEST-STUB SAFE + KEEP ALL TEST IDS + EMPTY-MATCH FALLBACK)
#
# Output (kompatibel STAGE 4):
# - MATCH_MAN_TRAIN_PATH: manifest_match_train_all.csv
# - MATCH_MAN_TEST_PATH : manifest_match_test.csv
# - MATCH_ROOT/match_summary.json
# - (tambahan) MATCH_ROOT/match_features_train_all.csv + match_features_test.csv
#
# Upgrade dari v6.2:
# 1) TEST_STUB safe: test case tetap lengkap walau feat hilang
# 2) Tidak drop feat_exists==0 untuk TEST (default: tulis empty match .npz)
# 3) Build list berbasis df_train_all/df_test (bukan man_* filtered) -> urutan & coverage aman
# 4) Numpy topk: fix argpartition bug (kth harus scalar)
# 5) Skip rows di feature CSV bisa (opsional) isi scalars dari file match yg sudah ada
# 6) Guard topk adaptif per N (lebih stabil & lebih cepat untuk grid kecil)
# ============================================================

import os, gc, time, json, math, hashlib
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd

# ----------------------------
# torch optional
# ----------------------------
_HAS_TORCH = False
try:
    import torch
    import torch.nn.functional as F
    _HAS_TORCH = True
    try:
        torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
    except Exception:
        pass
except Exception:
    torch = None
    F = None
    _HAS_TORCH = False

print("Torch available:", _HAS_TORCH)

# ----------------------------
# REQUIRE (sesuai STAGE 1)
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu.")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

# normalisasi dtype id
if "sample_id" in df_train_all.columns:
    df_train_all["sample_id"] = df_train_all["sample_id"].astype(str)
if "case_id" in df_train_all.columns:
    df_train_all["case_id"] = df_train_all["case_id"].astype(str)
if "case_id" in df_test.columns:
    df_test["case_id"] = df_test["case_id"].astype(str)
if "image_path" not in df_test.columns:
    df_test["image_path"] = ""

# ----------------------------
# TEST_STUB behavior
# ----------------------------
TEST_STUB = bool(globals().get("TEST_STUB", False))
if not TEST_STUB:
    try:
        n_test = int(len(df_test))
        hasf = int(df_test["image_path"].map(lambda p: Path(str(p)).exists()).sum())
        TEST_STUB = bool((n_test <= 5) or (hasf < n_test))
    except Exception:
        TEST_STUB = False

# default: untuk test, kalau feat hilang -> tulis empty match file agar STAGE 4 aman
WRITE_EMPTY_MATCH_FOR_MISSING_TEST  = int(globals().get("WRITE_EMPTY_MATCH_FOR_MISSING_TEST", 1))
WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN = int(globals().get("WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN", 0))

if TEST_STUB:
    WRITE_EMPTY_MATCH_FOR_MISSING_TEST = 1

print("TEST_STUB:", TEST_STUB)
print("WRITE_EMPTY_MATCH_FOR_MISSING_TEST :", int(WRITE_EMPTY_MATCH_FOR_MISSING_TEST))
print("WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN:", int(WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN))

# ----------------------------
# Auto-find manifests from STAGE 2
# ----------------------------
def _find_latest_file(pattern: str, roots):
    cands = []
    for r in roots:
        r = Path(r)
        if not r.exists():
            continue
        cands += list(r.rglob(pattern))
    cands = [p for p in cands if p.is_file()]
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

CACHE_SEARCH_ROOTS = [
    "/kaggle/working/recodai_luc/cache",
    "/kaggle/working/recodai_luc/cache/dino_v2_base",
    "/kaggle/working/recodai_luc/cache/dino_v2",
]

if "MAN_TRAIN_PATH" in globals():
    MAN_TRAIN_PATH = Path(str(MAN_TRAIN_PATH))
else:
    MAN_TRAIN_PATH = _find_latest_file("manifest_train_all.csv", CACHE_SEARCH_ROOTS)
    if MAN_TRAIN_PATH is None:
        MAN_TRAIN_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv")

if "MAN_TEST_PATH" in globals():
    MAN_TEST_PATH = Path(str(MAN_TEST_PATH))
else:
    MAN_TEST_PATH = _find_latest_file("manifest_test.csv", CACHE_SEARCH_ROOTS)
    if MAN_TEST_PATH is None:
        MAN_TEST_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv")

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train tidak ditemukan: {MAN_TRAIN_PATH} (jalankan STAGE 2 dulu).")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test tidak ditemukan: {MAN_TEST_PATH} (jalankan STAGE 2 dulu).")

man_train = pd.read_csv(MAN_TRAIN_PATH, dtype=str, low_memory=False)
man_test  = pd.read_csv(MAN_TEST_PATH,  dtype=str, low_memory=False)

need_train_cols = {"sample_id","feat_path","feat_exists"}
need_test_cols  = {"case_id","feat_path","feat_exists"}
if not need_train_cols.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all.csv missing cols: {need_train_cols - set(man_train.columns)}")
if not need_test_cols.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test.csv missing cols: {need_test_cols - set(man_test.columns)}")

man_train["sample_id"] = man_train["sample_id"].astype(str)
man_test["case_id"] = man_test["case_id"].astype(str)
man_train["feat_exists"] = man_train["feat_exists"].astype(int)
man_test["feat_exists"]  = man_test["feat_exists"].astype(int)

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows:", len(man_train), "| feat_exists=1:", int((man_train["feat_exists"]==1).sum()))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows:", len(man_test),  "| feat_exists=1:", int((man_test["feat_exists"]==1).sum()))

# optional: read cfg.json id (info only)
DINO_CFG_ID = ""
try:
    cfg_path = MAN_TRAIN_PATH.parent / "cfg.json"
    if cfg_path.exists():
        with open(cfg_path, "r", encoding="utf-8") as f:
            _cfg = json.load(f)
        # biasanya cfg_id tidak disimpan di cfg.json, tapi folder cfg_xxx sudah cukup
        DINO_CFG_ID = str(MAN_TRAIN_PATH.parent.name)
except Exception:
    DINO_CFG_ID = ""

# ----------------------------
# Config (match) + versioning
# ----------------------------
CFG_MATCH = {
    # core
    "topk": 80,
    "sim_thr": 0.78,
    "min_sep": 4,
    "close_metric": "max",   # "max" / "both"

    # distinctiveness filter (ratio/margin) — sim_scaled=(sim+1)/2
    "ratio_thr": 1.05,
    "margin_thr": 0.02,

    # voting
    "bin_step": 1,
    "peaks_M": 3,
    "min_peak_count": "auto",  # int atau "auto"
    "min_peak_frac": 0.03,
    "min_peak_min": 10,
    "peak_min_sep": 2,

    # weights
    "weight_power": 2.0,
    "margin_power": 1.0,

    # perf / io
    "skip_if_exists": True,
    "print_every": 200,
    "limit_train": None,
    "limit_test": None,

    # OPTIONAL speed knob (butuh torch)
    "proj_dim": None,
    "proj_seed": 2025,

    # internal
    "chunk": 256,            # sim chunk
    "topk_adapt": True,      # NEW v6.3
}

_match_blob = json.dumps(CFG_MATCH, sort_keys=True).encode("utf-8")
MATCH_CFG_ID = hashlib.sha1(_match_blob).hexdigest()[:12]

MATCH_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"match_base_cfg_{MATCH_CFG_ID}"
MATCH_TRAIN_DIR = MATCH_ROOT / "train_all"
MATCH_TEST_DIR  = MATCH_ROOT / "test"
MATCH_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
MATCH_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MATCH_CFG_ID :", MATCH_CFG_ID)
print("MATCH_ROOT   :", MATCH_ROOT)

# ----------------------------
# Helpers: load DINO cache (STAGE 2 format)
# ----------------------------
def load_dino_patch_from_npz(npz_path: str):
    p = Path(str(npz_path))
    if (not npz_path) or (str(npz_path).lower() == "nan") or (not p.exists()):
        return None
    z = np.load(p, allow_pickle=False)

    if "patch_desc" in z.files:
        pdsc = z["patch_desc"]
        if pdsc.ndim != 3:
            return None
        gh, gw, D = pdsc.shape
        feat = pdsc.reshape(gh*gw, D).astype(np.float32, copy=False)
        return feat, int(gh), int(gw)

    # fallback legacy
    if "feat" in z.files and "grid_h" in z.files and "grid_w" in z.files:
        feat = z["feat"].astype(np.float32, copy=False)
        gh = int(z["grid_h"]); gw = int(z["grid_w"])
        return feat, gh, gw

    return None

def save_match_npz(out_path: Path, payload: dict):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(out_path), **payload)

def read_match_cfg_id(npz_path: Path):
    try:
        z = np.load(str(npz_path), allow_pickle=False)
        if "match_cfg_id" in z.files:
            v = z["match_cfg_id"]
            return str(v.item()) if hasattr(v, "item") else str(v)
    except Exception:
        pass
    return ""

def read_match_scalars(npz_path: Path):
    """Untuk isi feature CSV saat skip (lebih akurat, overhead kecil)."""
    try:
        z = np.load(str(npz_path), allow_pickle=False)
        out = {}
        for k in ["has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
                  "n_pairs_thr","n_pairs_mnn","best_inlier_ratio","best_weight_frac",
                  "n_peaks","peak_entropy","best_dy","best_dx"]:
            if k in z.files:
                v = z[k]
                out[k] = float(v) if np.ndim(v)==0 and k not in ["has_peak","best_count","n_pairs_thr","n_pairs_mnn","n_peaks","best_dy","best_dx"] else int(v) if np.ndim(v)==0 else None
        return out
    except Exception:
        return {}

@lru_cache(maxsize=256)
def _grid_rc_np(gh: int, gw: int):
    rr = np.repeat(np.arange(gh, dtype=np.int32), gw)
    cc = np.tile(np.arange(gw, dtype=np.int32), gh)
    return rr, cc

def _is_close(src_idx: np.ndarray, dst_idx: np.ndarray, gh: int, gw: int, min_sep: int, metric: str):
    rr, cc = _grid_rc_np(gh, gw)
    dr = np.abs(rr[src_idx] - rr[dst_idx])
    dc = np.abs(cc[src_idx] - cc[dst_idx])
    if metric == "both":
        return (dr < min_sep) & (dc < min_sep)
    return (np.maximum(dr, dc) < min_sep)

def canonicalize_offsets(dy: np.ndarray, dx: np.ndarray):
    dy = dy.astype(np.int32, copy=False)
    dx = dx.astype(np.int32, copy=False)
    neg = (dy < 0) | ((dy == 0) & (dx < 0))
    dy2 = dy.copy(); dx2 = dx.copy()
    dy2[neg] = -dy2[neg]
    dx2[neg] = -dx2[neg]
    return dy2, dx2

def _encode_key(dy: np.ndarray, dx: np.ndarray):
    return (dy.astype(np.int64) << 32) ^ (dx.astype(np.int64) & np.int64(0xffffffff))

def _decode_key(key64: int):
    dy_pk = np.int32(key64 >> 32)
    dx_pk = np.int32(key64 & np.int64(0xffffffff))
    if dx_pk >= 2**31:
        dx_pk = dx_pk - 2**32
    return int(dy_pk), int(dx_pk)

def _adapt_topk(N: int, topk: int):
    # stabil & cepat: untuk N kecil, tidak maksa topk besar
    if N <= 0:
        return max(2, int(topk))
    k0 = min(max(2, int(topk)), N-1)
    k_adapt = max(16, int(round(2.0 * math.sqrt(N))))
    k = min(k0, k_adapt)
    return max(2, min(k, N-1))

# ----------------------------
# Optional random projection (torch only)
# ----------------------------
_PROJ_W = None
_PROJ_DIN = None

def maybe_project_torch(feat_t: "torch.Tensor", proj_dim: int, seed: int):
    global _PROJ_W, _PROJ_DIN
    if (not _HAS_TORCH) or proj_dim is None:
        return feat_t
    proj_dim = int(proj_dim)
    if proj_dim <= 0:
        return feat_t
    D = int(feat_t.shape[1])
    if _PROJ_W is None or _PROJ_DIN != D or int(_PROJ_W.shape[1]) != proj_dim:
        g = torch.Generator(device="cpu")
        g.manual_seed(int(seed))
        W = torch.randn((D, proj_dim), generator=g, dtype=torch.float32)
        W = F.normalize(W, dim=0)
        _PROJ_W = W.contiguous()
        _PROJ_DIN = D
    x = feat_t @ _PROJ_W
    x = F.normalize(x, dim=1)
    return x

# ----------------------------
# Robust matching (torch path)
# ----------------------------
def robust_match_one_torch(feat_np: np.ndarray, gh: int, gw: int, cfg: dict):
    N = int(gh * gw)
    topk = int(cfg["topk"])
    if bool(cfg.get("topk_adapt", True)):
        k = _adapt_topk(N, topk)
    else:
        k = min(max(2, topk), N-1)

    sim_thr = float(cfg["sim_thr"])
    min_sep = int(cfg["min_sep"])
    metric  = str(cfg.get("close_metric", "max"))
    ratio_thr  = float(cfg["ratio_thr"])
    margin_thr = float(cfg["margin_thr"])

    bin_step = int(cfg["bin_step"])
    peaks_M = int(cfg["peaks_M"])
    min_peak_count_cfg = cfg["min_peak_count"]
    peak_min_sep = int(cfg["peak_min_sep"])
    wpow = float(cfg["weight_power"])
    mpow = float(cfg["margin_power"])

    feat_np = np.ascontiguousarray(feat_np, dtype=np.float32)
    f = torch.from_numpy(feat_np).to(torch.float32)
    f = F.normalize(f, dim=1)

    # optional projection
    f = maybe_project_torch(f, cfg.get("proj_dim", None), cfg.get("proj_seed", 2025))

    idxs = torch.empty((N, k), dtype=torch.int64)
    vals = torch.empty((N, k), dtype=torch.float32)

    CHUNK = int(cfg.get("chunk", 256))
    CHUNK = max(32, min(CHUNK, N))

    for i0 in range(0, N, CHUNK):
        i1 = min(N, i0 + CHUNK)
        sim_chunk = f[i0:i1] @ f.T  # [B,N]
        rows = torch.arange(i0, i1, dtype=torch.int64)
        sim_chunk[torch.arange(i1-i0), rows] = -1e9
        v, ix = torch.topk(sim_chunk, k=k, dim=1, largest=True, sorted=True)
        vals[i0:i1] = v
        idxs[i0:i1] = ix

    # validity: far + sim>=thr
    src_idx = torch.arange(N, dtype=torch.int64).unsqueeze(1).expand(N, k)
    src_np = src_idx.reshape(-1).cpu().numpy().astype(np.int32, copy=False)
    dst_np = idxs.reshape(-1).cpu().numpy().astype(np.int32, copy=False)
    close_np = _is_close(src_np, dst_np, gh, gw, min_sep=min_sep, metric=metric)
    close = torch.from_numpy(close_np.reshape(N, k))

    valid = (~close) & (vals >= sim_thr)
    has_any = valid.any(dim=1)

    vals_valid = vals.masked_fill(~valid, -1e9)
    best_j = torch.argmax(vals_valid, dim=1)
    chosen_sim = vals_valid.gather(1, best_j.unsqueeze(1)).squeeze(1)
    chosen_dst = idxs.gather(1, best_j.unsqueeze(1)).squeeze(1)

    # top2 untuk ratio/margin (scaled)
    top2v = torch.topk(vals_valid, k=min(2, k), dim=1, largest=True, sorted=True).values
    v1 = top2v[:, 0]
    v2 = top2v[:, 1] if top2v.shape[1] > 1 else torch.full_like(v1, -1e9)

    s1 = (v1 + 1.0) * 0.5
    s2 = (v2 + 1.0) * 0.5
    only_one = v2 < -1e8
    ratio  = torch.where(only_one, torch.full_like(s1, 1e9), s1 / (s2 + 1e-9))
    margin = torch.where(only_one, torch.full_like(s1, 1e9), s1 - s2)

    keep = has_any & (v1 > -1e8) & (ratio >= ratio_thr) & (margin >= margin_thr)

    n_pairs_thr = int((vals >= sim_thr).sum().item())
    if keep.sum().item() == 0:
        return None, n_pairs_thr, 0

    src_keep = torch.nonzero(keep, as_tuple=False).squeeze(1).cpu().numpy().astype(np.int32)
    dst_keep = chosen_dst[keep].cpu().numpy().astype(np.int32)
    sim_keep = chosen_sim[keep].cpu().numpy().astype(np.float32)
    mar_keep = margin[keep].cpu().numpy().astype(np.float32)

    P = len(src_keep)
    if P == 0:
        return None, n_pairs_thr, 0

    # TRUE MNN
    order = np.lexsort((-sim_keep, dst_keep))
    dst_s = dst_keep[order]
    src_s = src_keep[order]
    first = np.ones(P, dtype=bool)
    first[1:] = (dst_s[1:] != dst_s[:-1])
    best_dst = dst_s[first]
    best_src = src_s[first]

    dst_best_src = np.full((N,), -1, dtype=np.int32)
    dst_best_src[best_dst] = best_src
    mnn_mask = (dst_best_src[dst_keep] == src_keep)

    src_mnn = src_keep[mnn_mask]
    dst_mnn = dst_keep[mnn_mask]
    sim_mnn = sim_keep[mnn_mask]
    mar_mnn = mar_keep[mnn_mask]

    n_pairs_mnn = int(len(src_mnn))
    if n_pairs_mnn == 0:
        return None, n_pairs_thr, 0

    # offsets
    src_r = src_mnn // gw; src_c = src_mnn % gw
    dst_r = dst_mnn // gw; dst_c = dst_mnn % gw
    dy = (dst_r - src_r).astype(np.int32)
    dx = (dst_c - src_c).astype(np.int32)
    dyc, dxc = canonicalize_offsets(dy, dx)

    if bin_step > 1:
        dyb = (np.round(dyc / bin_step)).astype(np.int32) * bin_step
        dxb = (np.round(dxc / bin_step)).astype(np.int32) * bin_step
    else:
        dyb, dxb = dyc, dxc

    # weights
    w = np.clip(sim_mnn, 0.0, 1.0).astype(np.float32, copy=False)
    if wpow != 1.0:
        w = np.power(w, wpow, dtype=np.float32)
    m = np.clip(mar_mnn, 0.0, None).astype(np.float32, copy=False)
    if mpow != 1.0:
        m = np.power(m, mpow, dtype=np.float32)
    wv = (w * (m + 1e-9)).astype(np.float32, copy=False)

    keys = _encode_key(dyb, dxb)
    uniq, inv = np.unique(keys, return_inverse=True)
    sum_w = np.bincount(inv, weights=wv, minlength=len(uniq)).astype(np.float32)
    cnt   = np.bincount(inv, minlength=len(uniq)).astype(np.int32)

    # min_peak_count
    if isinstance(min_peak_count_cfg, str) and min_peak_count_cfg.lower() == "auto":
        min_peak_count = max(int(cfg.get("min_peak_min", 10)), int(cfg.get("min_peak_frac", 0.03) * n_pairs_mnn))
    else:
        min_peak_count = int(min_peak_count_cfg)

    order_pk = np.argsort(-sum_w)

    # Peak NMS
    peaks = []
    for idx in order_pk:
        if cnt[idx] < min_peak_count:
            continue
        dy_pk, dx_pk = _decode_key(int(uniq[idx]))
        ok = True
        for (pdy, pdx, _, _) in peaks:
            if max(abs(dy_pk - pdy), abs(dx_pk - pdx)) < peak_min_sep:
                ok = False
                break
        if not ok:
            continue
        peaks.append((dy_pk, dx_pk, float(sum_w[idx]), int(cnt[idx])))
        if len(peaks) >= peaks_M:
            break

    if len(peaks) == 0:
        return None, n_pairs_thr, n_pairs_mnn

    return (src_mnn, dst_mnn, sim_mnn.astype(np.float32), keys, uniq, inv, sum_w, cnt, peaks), n_pairs_thr, n_pairs_mnn

# ----------------------------
# Robust matching (numpy fallback)
# ----------------------------
def _np_l2norm(x, axis=1, eps=1e-12):
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (n + eps)

def robust_match_one_numpy(feat_np: np.ndarray, gh: int, gw: int, cfg: dict):
    N = int(gh * gw)
    topk = int(cfg["topk"])
    if bool(cfg.get("topk_adapt", True)):
        k = _adapt_topk(N, topk)
    else:
        k = min(max(2, topk), N-1)

    sim_thr = float(cfg["sim_thr"])
    min_sep = int(cfg["min_sep"])
    metric  = str(cfg.get("close_metric", "max"))
    ratio_thr  = float(cfg["ratio_thr"])
    margin_thr = float(cfg["margin_thr"])

    bin_step = int(cfg["bin_step"])
    peaks_M = int(cfg["peaks_M"])
    min_peak_count_cfg = cfg["min_peak_count"]
    peak_min_sep = int(cfg["peak_min_sep"])
    wpow = float(cfg["weight_power"])
    mpow = float(cfg["margin_power"])

    feat_np = np.ascontiguousarray(feat_np, dtype=np.float32)
    f = _np_l2norm(feat_np, axis=1)

    CHUNK = int(cfg.get("chunk", 256))
    CHUNK = max(32, min(CHUNK, N))

    idxs = np.empty((N, k), dtype=np.int32)
    vals = np.empty((N, k), dtype=np.float32)

    for i0 in range(0, N, CHUNK):
        i1 = min(N, i0 + CHUNK)
        sim = f[i0:i1] @ f.T  # [B,N]
        for bi, gi in enumerate(range(i0, i1)):
            sim[bi, gi] = -1e9

        # FIX v6.3: argpartition kth harus scalar
        part = np.argpartition(-sim, kth=k-1, axis=1)[:, :k]
        part_vals = np.take_along_axis(sim, part, axis=1)
        ord2 = np.argsort(-part_vals, axis=1)
        ix = np.take_along_axis(part, ord2, axis=1)
        v  = np.take_along_axis(sim, ix, axis=1)

        idxs[i0:i1] = ix.astype(np.int32, copy=False)
        vals[i0:i1] = v.astype(np.float32, copy=False)

    # validity: far + sim>=thr
    src_rep = np.repeat(np.arange(N, dtype=np.int32), k)
    dst_rep = idxs.reshape(-1)
    close = _is_close(src_rep, dst_rep, gh, gw, min_sep=min_sep, metric=metric).reshape(N, k)

    valid = (~close) & (vals >= sim_thr)
    has_any = valid.any(axis=1)

    vals_valid = np.where(valid, vals, -1e9)
    best_j = np.argmax(vals_valid, axis=1)
    chosen_sim = vals_valid[np.arange(N), best_j]
    chosen_dst = idxs[np.arange(N), best_j]

    # top2 valid
    top2_ix = np.argpartition(-vals_valid, kth=min(1, k-1), axis=1)[:, :min(2, k)]
    top2_v  = np.take_along_axis(vals_valid, top2_ix, axis=1)
    ord2 = np.argsort(-top2_v, axis=1)
    top2_v = np.take_along_axis(top2_v, ord2, axis=1)

    v1 = top2_v[:, 0]
    v2 = top2_v[:, 1] if top2_v.shape[1] > 1 else np.full_like(v1, -1e9)

    s1 = (v1 + 1.0) * 0.5
    s2 = (v2 + 1.0) * 0.5
    only_one = v2 < -1e8
    ratio  = np.where(only_one, 1e9, s1 / (s2 + 1e-9))
    margin = np.where(only_one, 1e9, s1 - s2)

    keep = has_any & (v1 > -1e8) & (ratio >= ratio_thr) & (margin >= margin_thr)
    n_pairs_thr = int((vals >= sim_thr).sum())

    if keep.sum() == 0:
        return None, n_pairs_thr, 0

    src_keep = np.where(keep)[0].astype(np.int32)
    dst_keep = chosen_dst[keep].astype(np.int32)
    sim_keep = chosen_sim[keep].astype(np.float32)
    mar_keep = margin[keep].astype(np.float32)

    P = len(src_keep)
    if P == 0:
        return None, n_pairs_thr, 0

    # TRUE MNN
    order = np.lexsort((-sim_keep, dst_keep))
    dst_s = dst_keep[order]
    src_s = src_keep[order]

    first = np.ones(P, dtype=bool)
    first[1:] = (dst_s[1:] != dst_s[:-1])
    best_dst = dst_s[first]
    best_src = src_s[first]

    dst_best_src = np.full((N,), -1, dtype=np.int32)
    dst_best_src[best_dst] = best_src
    mnn_mask = (dst_best_src[dst_keep] == src_keep)

    src_mnn = src_keep[mnn_mask]
    dst_mnn = dst_keep[mnn_mask]
    sim_mnn = sim_keep[mnn_mask]
    mar_mnn = mar_keep[mnn_mask]

    n_pairs_mnn = int(len(src_mnn))
    if n_pairs_mnn == 0:
        return None, n_pairs_thr, 0

    # offsets
    src_r = src_mnn // gw; src_c = src_mnn % gw
    dst_r = dst_mnn // gw; dst_c = dst_mnn % gw
    dy = (dst_r - src_r).astype(np.int32)
    dx = (dst_c - src_c).astype(np.int32)
    dyc, dxc = canonicalize_offsets(dy, dx)

    if bin_step > 1:
        dyb = (np.round(dyc / bin_step)).astype(np.int32) * bin_step
        dxb = (np.round(dxc / bin_step)).astype(np.int32) * bin_step
    else:
        dyb, dxb = dyc, dxc

    # weights
    w = np.clip(sim_mnn, 0.0, 1.0).astype(np.float32, copy=False)
    if wpow != 1.0:
        w = np.power(w, wpow, dtype=np.float32)
    m = np.clip(mar_mnn, 0.0, None).astype(np.float32, copy=False)
    if mpow != 1.0:
        m = np.power(m, mpow, dtype=np.float32)
    wv = (w * (m + 1e-9)).astype(np.float32, copy=False)

    keys = _encode_key(dyb, dxb)
    uniq, inv = np.unique(keys, return_inverse=True)
    sum_w = np.bincount(inv, weights=wv, minlength=len(uniq)).astype(np.float32)
    cnt   = np.bincount(inv, minlength=len(uniq)).astype(np.int32)

    if isinstance(min_peak_count_cfg, str) and min_peak_count_cfg.lower() == "auto":
        min_peak_count = max(int(cfg.get("min_peak_min", 10)), int(cfg.get("min_peak_frac", 0.03) * n_pairs_mnn))
    else:
        min_peak_count = int(min_peak_count_cfg)

    order_pk = np.argsort(-sum_w)

    peaks = []
    for idx in order_pk:
        if cnt[idx] < min_peak_count:
            continue
        dy_pk, dx_pk = _decode_key(int(uniq[idx]))
        ok = True
        for (pdy, pdx, _, _) in peaks:
            if max(abs(dy_pk - pdy), abs(dx_pk - pdx)) < peak_min_sep:
                ok = False
                break
        if not ok:
            continue
        peaks.append((dy_pk, dx_pk, float(sum_w[idx]), int(cnt[idx])))
        if len(peaks) >= peaks_M:
            break

    if len(peaks) == 0:
        return None, n_pairs_thr, n_pairs_mnn

    return (src_mnn, dst_mnn, sim_mnn.astype(np.float32), keys, uniq, inv, sum_w, cnt, peaks), n_pairs_thr, n_pairs_mnn

# ----------------------------
# Wrapper: robust_match_one
# ----------------------------
def _empty_payload(gh: int, gw: int, n_pairs_thr=0, n_pairs_mnn=0):
    return {
        "grid_h": np.int32(gh),
        "grid_w": np.int32(gw),
        "has_peak": np.int8(0),
        "peak_ratio": np.float32(0.0),
        "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
        "peaks_weight": np.zeros((0,), dtype=np.float32),
        "peaks_count": np.zeros((0,), dtype=np.int32),
        "best_src": np.zeros((0,), dtype=np.int32),
        "best_dst": np.zeros((0,), dtype=np.int32),
        "best_sim": np.zeros((0,), dtype=np.float16),
        "best_weight": np.float32(0.0),
        "best_count": np.int32(0),
        "best_mean_sim": np.float32(0.0),
        "n_pairs_thr": np.int32(n_pairs_thr),
        "n_pairs_mnn": np.int32(n_pairs_mnn),
        "best_inlier_ratio": np.float32(0.0),
        "best_weight_frac": np.float32(0.0),
        "n_peaks": np.int32(0),
        "peak_entropy": np.float32(0.0),
        "best_dy": np.int16(0),
        "best_dx": np.int16(0),
    }

def robust_match_one(feat_np: np.ndarray, gh: int, gw: int, cfg: dict):
    N = int(gh * gw)
    if feat_np is None or feat_np.ndim != 2 or feat_np.shape[0] != N or N < 4:
        return _empty_payload(int(gh), int(gw), 0, 0)

    if _HAS_TORCH:
        core, n_pairs_thr, n_pairs_mnn = robust_match_one_torch(feat_np, gh, gw, cfg)
    else:
        core, n_pairs_thr, n_pairs_mnn = robust_match_one_numpy(feat_np, gh, gw, cfg)

    if core is None:
        return _empty_payload(int(gh), int(gw), int(n_pairs_thr), int(n_pairs_mnn))

    src_mnn, dst_mnn, sim_mnn, keys, uniq, inv, sum_w, cnt, peaks = core

    peaks_arr   = np.array([[p[0], p[1]] for p in peaks], dtype=np.int16)
    weights_arr = np.array([p[2] for p in peaks], dtype=np.float32)
    counts_arr  = np.array([p[3] for p in peaks], dtype=np.int32)

    w1 = peaks[0][2]
    w2 = peaks[1][2] if len(peaks) > 1 else 0.0
    peak_ratio = float(w1 / (w2 + 1e-9)) if w2 > 0 else float(1e9)

    best_dy, best_dx, best_weight, best_count = peaks[0]
    best_key = _encode_key(np.array([best_dy], dtype=np.int32), np.array([best_dx], dtype=np.int32))[0]
    same = (keys == best_key)

    best_src = src_mnn[same].astype(np.int32, copy=False)
    best_dst = dst_mnn[same].astype(np.int32, copy=False)
    best_sim = sim_mnn[same].astype(np.float16, copy=False)
    best_mean_sim = float(np.mean(sim_mnn[same])) if best_src.size > 0 else 0.0

    total_w = float(np.sum(sum_w)) + 1e-9
    best_inlier_ratio = float(best_count / max(1, int(len(src_mnn))))
    best_weight_frac  = float(best_weight / total_w)

    pw = weights_arr / (weights_arr.sum() + 1e-9)
    peak_entropy = float(-(pw * np.log(pw + 1e-12)).sum()) if len(pw) else 0.0

    return {
        "grid_h": np.int32(gh),
        "grid_w": np.int32(gw),
        "has_peak": np.int8(1),
        "peak_ratio": np.float32(peak_ratio),
        "peaks_dy_dx": peaks_arr,
        "peaks_weight": weights_arr,
        "peaks_count": counts_arr,
        "best_src": best_src,
        "best_dst": best_dst,
        "best_sim": best_sim,
        "best_weight": np.float32(best_weight),
        "best_count": np.int32(best_count),
        "best_mean_sim": np.float32(best_mean_sim),
        "n_pairs_thr": np.int32(n_pairs_thr),
        "n_pairs_mnn": np.int32(int(len(src_mnn))),
        "best_inlier_ratio": np.float32(best_inlier_ratio),
        "best_weight_frac": np.float32(best_weight_frac),
        "n_peaks": np.int32(len(peaks)),
        "peak_entropy": np.float32(peak_entropy),
        "best_dy": np.int16(best_dy),
        "best_dx": np.int16(best_dx),
    }

# ----------------------------
# Build processing lists (SAFE: base on df_train_all/df_test)
# ----------------------------
# base cols
train_base_cols = [c for c in ["sample_id","case_id","variant","fold","y_forged","image_path"] if c in df_train_all.columns]
test_base_cols  = [c for c in ["case_id","image_path"] if c in df_test.columns]

train_base = df_train_all[train_base_cols].copy()
train_base["sample_id"] = train_base["sample_id"].astype(str)
train_base = train_base.drop_duplicates(subset=["sample_id"], keep="first")

test_base = df_test[test_base_cols].copy()
test_base["case_id"] = test_base["case_id"].astype(str)
test_base = test_base.drop_duplicates(subset=["case_id"], keep="first")

# merge manifest (left join -> keep all ids)
train_list = train_base.merge(
    man_train[["sample_id","feat_path","feat_exists"]].copy(),
    on="sample_id", how="left"
)
test_list = test_base.merge(
    man_test[["case_id","feat_path","feat_exists"]].copy(),
    on="case_id", how="left"
)

train_list["feat_exists"] = train_list["feat_exists"].fillna(0).astype(int)
test_list["feat_exists"]  = test_list["feat_exists"].fillna(0).astype(int)
train_list["feat_path"] = train_list["feat_path"].fillna("").astype(str)
test_list["feat_path"]  = test_list["feat_path"].fillna("").astype(str)

# sort
sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in train_list.columns]
if sort_train_cols:
    train_list = train_list.sort_values(sort_train_cols).reset_index(drop=True)
else:
    train_list = train_list.sort_values(["sample_id"]).reset_index(drop=True)
test_list = test_list.sort_values(["case_id"]).reset_index(drop=True)

# limits
if CFG_MATCH["limit_train"] is not None:
    train_list = train_list.iloc[:int(CFG_MATCH["limit_train"])].copy()
if CFG_MATCH["limit_test"] is not None:
    test_list = test_list.iloc[:int(CFG_MATCH["limit_test"])].copy()

print("\nTo process:")
print(f"  train samples: {len(train_list):,} (all sample_id, feat_exists=1: {int((train_list['feat_exists']==1).sum()):,})")
print(f"  test  cases  : {len(test_list):,} (all case_id,  feat_exists=1: {int((test_list['feat_exists']==1).sum()):,})")
print(f"  proj_dim     : {CFG_MATCH.get('proj_dim', None)} (torch-only, None=OFF)")

# ----------------------------
# Run matching
# ----------------------------
FEATURE_COLS = [
    "uid","id_col","y_forged","feat_exists","match_exists","skipped",
    "has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
    "n_pairs_thr","n_pairs_mnn","best_inlier_ratio","best_weight_frac",
    "n_peaks","peak_entropy","best_dy","best_dx"
]

def run_block(df_list: pd.DataFrame, id_col: str, out_dir: Path, label: str, write_empty_missing: bool):
    t0 = time.time()
    done = skipped = rebuilt = failed = empty_written = 0
    feats_rows = []

    cols = set(df_list.columns)

    for i, r in enumerate(df_list.itertuples(index=False), start=1):
        uid = str(getattr(r, id_col))
        feat_path = str(getattr(r, "feat_path", ""))
        if feat_path.lower() == "nan":
            feat_path = ""
        feat_exists = int(getattr(r, "feat_exists", 0))
        outp = out_dir / f"{uid}.npz"

        yv = getattr(r, "y_forged", None)
        yv_i = int(yv) if yv is not None and str(yv).strip() != "nan" else -1

        base_row = {c: 0 for c in FEATURE_COLS}
        base_row.update({"uid": uid, "id_col": id_col, "y_forged": yv_i, "feat_exists": feat_exists})

        if outp.exists() and CFG_MATCH["skip_if_exists"]:
            old = read_match_cfg_id(outp)
            if old == MATCH_CFG_ID:
                skipped += 1
                base_row.update({"match_exists": 1, "skipped": 1})
                # isi scalars dari file agar CSV tidak misleading
                sc = read_match_scalars(outp)
                for k in sc:
                    if k in base_row and sc[k] is not None:
                        base_row[k] = sc[k]
                feats_rows.append(base_row)
                continue
            else:
                try:
                    outp.unlink()
                except Exception:
                    pass
                rebuilt += 1

        loaded = load_dino_patch_from_npz(feat_path) if (feat_exists == 1) else None

        if loaded is None:
            # missing/invalid feature
            if write_empty_missing:
                try:
                    payload = _empty_payload(1, 1, 0, 0)
                    payload["match_cfg_id"] = np.array(MATCH_CFG_ID, dtype="<U32")
                    payload["match_version"] = np.array("v6.3", dtype="<U16")
                    payload["uid"] = np.array(uid, dtype="<U128")
                    if "case_id" in cols:
                        payload["case_id"] = np.array(str(getattr(r, "case_id", "")), dtype="<U64")
                    if "variant" in cols:
                        payload["variant"] = np.array(str(getattr(r, "variant", "")), dtype="<U32")
                    payload["empty_reason"] = np.array("missing_feat", dtype="<U32")
                    save_match_npz(outp, payload)

                    empty_written += 1
                    base_row.update({"match_exists": 1, "skipped": 0, "has_peak": 0})
                except Exception as e:
                    failed += 1
                    if failed <= 10:
                        print(f"[{label}] WARN empty-write FAIL uid={uid} err={type(e).__name__}: {e}")
                    base_row.update({"match_exists": 0, "skipped": 0})
            else:
                base_row.update({"match_exists": 0, "skipped": 0})
            feats_rows.append(base_row)
            continue

        feat_np, gh, gw = loaded

        try:
            payload = robust_match_one(feat_np, gh, gw, CFG_MATCH)

            payload["match_cfg_id"] = np.array(MATCH_CFG_ID, dtype="<U32")
            payload["match_version"] = np.array("v6.3", dtype="<U16")
            payload["uid"] = np.array(uid, dtype="<U128")
            if "case_id" in cols:
                payload["case_id"] = np.array(str(getattr(r, "case_id", "")), dtype="<U64")
            if "variant" in cols:
                payload["variant"] = np.array(str(getattr(r, "variant", "")), dtype="<U32")

            save_match_npz(outp, payload)
            done += 1

            base_row.update({
                "match_exists": 1,
                "skipped": 0,
                "has_peak": int(payload["has_peak"]),
                "peak_ratio": float(payload["peak_ratio"]),
                "best_weight": float(payload["best_weight"]),
                "best_count": int(payload["best_count"]),
                "best_mean_sim": float(payload["best_mean_sim"]),
                "n_pairs_thr": int(payload["n_pairs_thr"]),
                "n_pairs_mnn": int(payload["n_pairs_mnn"]),
                "best_inlier_ratio": float(payload["best_inlier_ratio"]),
                "best_weight_frac": float(payload["best_weight_frac"]),
                "n_peaks": int(payload["n_peaks"]),
                "peak_entropy": float(payload["peak_entropy"]),
                "best_dy": int(payload.get("best_dy", 0)),
                "best_dx": int(payload.get("best_dx", 0)),
            })
            feats_rows.append(base_row)

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")
            base_row.update({"match_exists": 0, "skipped": 0})
            feats_rows.append(base_row)

        if (done + skipped + empty_written) > 0 and ((done + skipped + empty_written) % int(CFG_MATCH["print_every"]) == 0):
            dt = time.time() - t0
            rate = (done + skipped + empty_written) / max(dt, 1e-9)
            print(f"[{label}] proc={done+skipped+empty_written:,}/{len(df_list):,} | done={done:,} skipped={skipped:,} empty={empty_written:,} rebuilt={rebuilt:,} failed={failed:,} | {rate:.2f} img/s | last={uid}")
            gc.collect()

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done        : {done:,}")
    print(f"  skipped     : {skipped:,}")
    print(f"  empty_write : {empty_written:,}")
    print(f"  rebuilt     : {rebuilt:,}")
    print(f"  failed      : {failed:,}")
    print(f"  time_s      : {dt:.1f}")
    return {"done": done, "skipped": skipped, "empty_written": empty_written, "rebuilt": rebuilt, "failed": failed, "time_s": dt}, pd.DataFrame(feats_rows)

print("\n[1/2] Matching TRAIN_ALL ...")
rep_train, df_feat_train = run_block(
    train_list, id_col="sample_id", out_dir=MATCH_TRAIN_DIR, label="train_all",
    write_empty_missing=bool(WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN)
)

print("\n[2/2] Matching TEST ...")
rep_test, df_feat_test = run_block(
    test_list, id_col="case_id", out_dir=MATCH_TEST_DIR, label="test",
    write_empty_missing=bool(WRITE_EMPTY_MATCH_FOR_MISSING_TEST)
)

# ----------------------------
# Write manifests for Stage 4 (SAFE: include all ids)
# ----------------------------
man_match_train = pd.DataFrame({
    "sample_id": train_list["sample_id"].astype(str).values,
    "case_id": train_list["case_id"].astype(str).fillna("").values if "case_id" in train_list.columns else np.array([""]*len(train_list)),
    "variant": train_list["variant"].astype(str).fillna("").values if "variant" in train_list.columns else np.array([""]*len(train_list)),
    "fold": train_list["fold"].fillna(-1).astype(int).values if "fold" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "y_forged": train_list["y_forged"].fillna(-1).astype(int).values if "y_forged" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "feat_exists": train_list["feat_exists"].fillna(0).astype(int).values,
    "match_path": [str(MATCH_TRAIN_DIR / f"{sid}.npz") for sid in train_list["sample_id"].astype(str).values],
})
man_match_train["match_exists"] = man_match_train["match_path"].map(lambda p: Path(p).exists()).astype(int)

man_match_test = pd.DataFrame({
    "case_id": test_list["case_id"].astype(str).values,
    "feat_exists": test_list["feat_exists"].fillna(0).astype(int).values,
    "match_path": [str(MATCH_TEST_DIR / f"{cid}.npz") for cid in test_list["case_id"].astype(str).values],
})
man_match_test["match_exists"] = man_match_test["match_path"].map(lambda p: Path(p).exists()).astype(int)

MATCH_MAN_TRAIN_PATH = MATCH_ROOT / "manifest_match_train_all.csv"
MATCH_MAN_TEST_PATH  = MATCH_ROOT / "manifest_match_test.csv"
man_match_train.to_csv(MATCH_MAN_TRAIN_PATH, index=False)
man_match_test.to_csv(MATCH_MAN_TEST_PATH, index=False)

# ----------------------------
# Save feature CSVs
# ----------------------------
FEAT_TRAIN_PATH = MATCH_ROOT / "match_features_train_all.csv"
FEAT_TEST_PATH  = MATCH_ROOT / "match_features_test.csv"
df_feat_train.to_csv(FEAT_TRAIN_PATH, index=False)
df_feat_test.to_csv(FEAT_TEST_PATH, index=False)

# quick AUC helper (tanpa sklearn)
def auc_roc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    m = np.isfinite(y_score) & (y_true >= 0)
    y_true = y_true[m]; y_score = y_score[m]
    if y_true.size == 0:
        return np.nan
    pos = (y_true == 1)
    neg = (y_true == 0)
    n_pos = int(pos.sum()); n_neg = int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, y_true.size + 1, dtype=float)

    s_sorted = y_score[order]
    r_sorted = ranks[order]
    i = 0
    while i < len(s_sorted):
        j = i + 1
        while j < len(s_sorted) and s_sorted[j] == s_sorted[i]:
            j += 1
        if j - i > 1:
            r_mean = r_sorted[i:j].mean()
            ranks[order[i:j]] = r_mean
        i = j

    sum_r_pos = ranks[pos].sum()
    auc = (sum_r_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg)
    return float(auc)

if "y_forged" in df_feat_train.columns and (df_feat_train["y_forged"] >= 0).any():
    for col in ["best_weight_frac", "best_inlier_ratio", "peak_ratio", "n_pairs_mnn"]:
        if col not in df_feat_train.columns:
            df_feat_train[col] = 0.0
    score = (
        df_feat_train["best_weight_frac"].fillna(0).astype(float) *
        (0.5 + df_feat_train["best_inlier_ratio"].fillna(0).astype(float)) *
        np.log1p(df_feat_train["peak_ratio"].fillna(0).astype(float))
    )
    auc = auc_roc(df_feat_train["y_forged"].values, score.values)
    print("\n[Eval] Rough AUC using match-score:", auc)

with open(MATCH_ROOT / "match_summary.json", "w") as f:
    json.dump({
        "cfg": CFG_MATCH,
        "match_cfg_id": MATCH_CFG_ID,
        "DINO_MAN_TRAIN_PATH": str(MAN_TRAIN_PATH),
        "DINO_MAN_TEST_PATH": str(MAN_TEST_PATH),
        "dino_cfg_folder": str(DINO_CFG_ID),
        "torch_available": bool(_HAS_TORCH),
        "test_stub": bool(TEST_STUB),
        "write_empty_missing": {
            "train": int(WRITE_EMPTY_MATCH_FOR_MISSING_TRAIN),
            "test": int(WRITE_EMPTY_MATCH_FOR_MISSING_TEST),
        },
        "train": rep_train,
        "test": rep_test,
        "outputs": {
            "MATCH_MAN_TRAIN_PATH": str(MATCH_MAN_TRAIN_PATH),
            "MATCH_MAN_TEST_PATH": str(MATCH_MAN_TEST_PATH),
            "FEAT_TRAIN_PATH": str(FEAT_TRAIN_PATH),
            "FEAT_TEST_PATH": str(FEAT_TEST_PATH),
        }
    }, f, indent=2)

print("\nWrote match manifests:")
print(" -", MATCH_MAN_TRAIN_PATH, "| exists_rate:", float(man_match_train["match_exists"].mean()) if len(man_match_train) else 0.0)
print(" -", MATCH_MAN_TEST_PATH,  "| exists_rate:", float(man_match_test["match_exists"].mean()) if len(man_match_test) else 0.0)
print("Extra feature CSV:")
print(" -", FEAT_TRAIN_PATH)
print(" -", FEAT_TEST_PATH)
print(" -", MATCH_ROOT / "match_summary.json")

MATCH_CACHE_ROOT = str(MATCH_ROOT)
print("\nDONE. Exported: CFG_MATCH, MATCH_CFG_ID, MATCH_CACHE_ROOT, MATCH_TRAIN_DIR, MATCH_TEST_DIR, MATCH_MAN_TRAIN_PATH, MATCH_MAN_TEST_PATH")

globals().update({
    "CFG_MATCH": CFG_MATCH,
    "MATCH_CFG_ID": MATCH_CFG_ID,
    "MATCH_CACHE_ROOT": MATCH_CACHE_ROOT,
    "MATCH_TRAIN_DIR": MATCH_TRAIN_DIR,
    "MATCH_TEST_DIR": MATCH_TEST_DIR,
    "MATCH_MAN_TRAIN_PATH": MATCH_MAN_TRAIN_PATH,
    "MATCH_MAN_TEST_PATH": MATCH_MAN_TEST_PATH,
    "FEAT_TRAIN_PATH": FEAT_TRAIN_PATH,
    "FEAT_TEST_PATH": FEAT_TEST_PATH,
    "TEST_STUB": bool(TEST_STUB),
})


# Verification, Mask Reconstruction & Postprocess (One Block)

In [ ]:
# ============================================================
# STAGE 4 — Verification, Mask Reconstruction & Postprocess (ONE CELL)
# REVISI FULL v7.3 (STAGE3 v6.3 compatible: KEEP ALL TEST IDS + empty match OK)
#
# Fix/upgrade dari v7.2:
# - Worklist dibangun dari df_train_all/df_test (coverage aman untuk semua id)
# - Tidak filter man_train/man_test feat_exists==1 (hindari hilang id saat join)
# - Tambah feat_exists/match_exists di scalars rows (debug lebih mudah)
# - Tetap: bugfix leak comb_s/cnt_g, re-init grid buffers, drop duplicates, skip scalars from npz
# ============================================================

import os, gc, time, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# REQUIRE
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu.")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

# ----------------------------
# Helper: normalize id -> string stabil
# ----------------------------
def _norm_one_id(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    if isinstance(x, (np.integer, int)):
        return str(int(x))
    if isinstance(x, (np.floating, float)):
        if np.isfinite(x) and abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(float(x))
    s = str(x)
    if s.endswith(".0"):
        head = s[:-2]
        if head.isdigit():
            return head
    return s

def norm_id_series(s: pd.Series) -> pd.Series:
    return s.map(_norm_one_id)

for c in ["sample_id", "case_id"]:
    if c in df_train_all.columns:
        df_train_all[c] = norm_id_series(df_train_all[c])
if "case_id" in df_test.columns:
    df_test["case_id"] = norm_id_series(df_test["case_id"])
if "image_path" not in df_test.columns:
    df_test["image_path"] = ""

# ----------------------------
# Auto-find manifests (STAGE 2 / STAGE 3)
# ----------------------------
def _auto_find_latest(root: Path, filename: str) -> Path:
    root = Path(root)
    if not root.exists():
        return Path(filename)
    cands = list(root.rglob(filename))
    cands = [p for p in cands if p.is_file()]
    if not cands:
        return Path(filename)
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

# STAGE 2 manifests
MAN_TRAIN_PATH = globals().get("MAN_TRAIN_PATH", None)
MAN_TEST_PATH  = globals().get("MAN_TEST_PATH", None)
MAN_TRAIN_PATH = Path(str(MAN_TRAIN_PATH)) if MAN_TRAIN_PATH is not None else Path("")
MAN_TEST_PATH  = Path(str(MAN_TEST_PATH)) if MAN_TEST_PATH is not None else Path("")

if not MAN_TRAIN_PATH.exists():
    MAN_TRAIN_PATH = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_train_all.csv")
if not MAN_TEST_PATH.exists():
    MAN_TEST_PATH  = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_test.csv")

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train STAGE 2 tidak ditemukan: {MAN_TRAIN_PATH}")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test STAGE 2 tidak ditemukan: {MAN_TEST_PATH}")

man_train = pd.read_csv(MAN_TRAIN_PATH, dtype=str, low_memory=False)
man_test  = pd.read_csv(MAN_TEST_PATH,  dtype=str, low_memory=False)

need2_train = {"sample_id","feat_path","feat_exists"}
need2_test  = {"case_id","feat_path","feat_exists"}
if not need2_train.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all missing: {need2_train - set(man_train.columns)}")
if not need2_test.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test missing: {need2_test - set(man_test.columns)}")

man_train["sample_id"] = norm_id_series(man_train["sample_id"])
man_test["case_id"]    = norm_id_series(man_test["case_id"])
man_train["feat_exists"] = man_train["feat_exists"].fillna("0").astype(int)
man_test["feat_exists"]  = man_test["feat_exists"].fillna("0").astype(int)

# STAGE 3 manifests
MATCH_MAN_TRAIN_PATH = globals().get("MATCH_MAN_TRAIN_PATH", None)
MATCH_MAN_TEST_PATH  = globals().get("MATCH_MAN_TEST_PATH", None)
MATCH_MAN_TRAIN_PATH = Path(str(MATCH_MAN_TRAIN_PATH)) if MATCH_MAN_TRAIN_PATH is not None else Path("")
MATCH_MAN_TEST_PATH  = Path(str(MATCH_MAN_TEST_PATH)) if MATCH_MAN_TEST_PATH is not None else Path("")

if not MATCH_MAN_TRAIN_PATH.exists():
    MATCH_MAN_TRAIN_PATH = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_match_train_all.csv")
if not MATCH_MAN_TEST_PATH.exists():
    MATCH_MAN_TEST_PATH  = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_match_test.csv")

if not MATCH_MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest match train STAGE 3 tidak ditemukan: {MATCH_MAN_TRAIN_PATH}")
if not MATCH_MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest match test STAGE 3 tidak ditemukan: {MATCH_MAN_TEST_PATH}")

man_match_train = pd.read_csv(MATCH_MAN_TRAIN_PATH, dtype=str, low_memory=False)
man_match_test  = pd.read_csv(MATCH_MAN_TEST_PATH,  dtype=str, low_memory=False)

need3_train = {"sample_id","match_path","match_exists"}
need3_test  = {"case_id","match_path","match_exists"}
if not need3_train.issubset(set(man_match_train.columns)):
    raise RuntimeError(f"manifest_match_train_all missing: {need3_train - set(man_match_train.columns)}")
if not need3_test.issubset(set(man_match_test.columns)):
    raise RuntimeError(f"manifest_match_test missing: {need3_test - set(man_match_test.columns)}")

man_match_train["sample_id"] = norm_id_series(man_match_train["sample_id"])
man_match_test["case_id"]    = norm_id_series(man_match_test["case_id"])
man_match_train["match_exists"] = man_match_train["match_exists"].fillna("0").astype(int)
man_match_test["match_exists"]  = man_match_test["match_exists"].fillna("0").astype(int)

# ----------------------------
# CONFIG
# ----------------------------
CFG_RECON = {
    "sim_inlier_thr": 0.825,
    "min_pairs": 22,
    "min_pairs_small": 14,
    "small_grid_N": 18*18,

    "score_mix": 0.70,
    "grid_thr_q": 0.90,
    "min_count_q": 0.78,
    "thr_grid_floor": 0.36,

    "grid_smooth_sigma": 0.85,
    "grid_open_ks": 0,
    "grid_close_ks": 3,

    "enable_relax_if_empty": True,
    "relax_q_to": 0.82,
    "relax_min_count_q_to": 0.60,

    "enable_dilate_if_tiny": True,
    "tiny_area_frac": 0.00025,
    "grid_dilate_ks": 3,

    "do_open": True,
    "open_ks": 3,
    "do_close": True,
    "close_ks": 3,
    "fill_holes": True,

    "min_area_frac": 0.00045,
    "keep_topk_components": 2,

    "max_area_frac": 0.30,
    "enable_tighten_overmask": True,
    "tighten_q_steps": [0.93, 0.95, 0.97, 0.985],
    "tighten_cntq_steps": [0.82, 0.86, 0.90],

    "skip_if_exists": True,
    "print_every": 400,
}
_CFG_HASH = hashlib.md5(json.dumps(CFG_RECON, sort_keys=True).encode()).hexdigest()[:10]

PRED_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"pred_base_v3_v7_cfg_{_CFG_HASH}"
PRED_TRAIN_DIR = PRED_ROOT / "train_all"
PRED_TEST_DIR  = PRED_ROOT / "test"
PRED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PRED_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows:", len(man_train), "| feat_exists=1:", int((man_train["feat_exists"]==1).sum()))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows:", len(man_test),  "| feat_exists=1:", int((man_test["feat_exists"]==1).sum()))
print("MATCH_MAN_TRAIN_PATH:", MATCH_MAN_TRAIN_PATH, "| rows:", len(man_match_train))
print("MATCH_MAN_TEST_PATH :", MATCH_MAN_TEST_PATH,  "| rows:", len(man_match_test))
print("PRED_ROOT:", PRED_ROOT)
print("SCIPY available:", _HAS_SCIPY)

# ----------------------------
# Helpers
# ----------------------------
def _meta_to_str(x):
    if isinstance(x, np.ndarray):
        try:
            x = x.item()
        except Exception:
            x = x.reshape(-1)[0]
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8", errors="ignore")
    return str(x)

def _read_meta_from_feat_npz(feat_path: str):
    p = Path(str(feat_path))
    if (not feat_path) or (str(feat_path).lower() == "nan") or (not p.exists()):
        return None
    z = np.load(p, allow_pickle=False)
    if "meta" not in z.files:
        return None
    try:
        meta = json.loads(_meta_to_str(z["meta"]))
    except Exception:
        return None

    oh, ow = meta.get("orig_hw", [None, None])
    rh_rw = meta.get("resized_hw_base", None)
    if rh_rw is None:
        rh_rw = meta.get("resized_hw", None)
    if rh_rw is None:
        rh, rw = None, None
    else:
        rh, rw = rh_rw[0], rh_rw[1]

    gh_gw = meta.get("grid_hw", [None, None])
    gh, gw = gh_gw[0], gh_gw[1]
    patch = meta.get("patch_size", None)

    if None in [oh, ow, rh, rw, gh, gw, patch]:
        return None
    return {
        "orig_h": int(oh), "orig_w": int(ow),
        "proc_h": int(rh), "proc_w": int(rw),
        "grid_h": int(gh), "grid_w": int(gw),
        "patch": int(patch),
    }

def _load_match_npz(match_path: str):
    p = Path(str(match_path))
    if (not match_path) or (str(match_path).lower() == "nan") or (not p.exists()):
        return None
    z = np.load(p, allow_pickle=False)

    def _s(v, default=0):
        if v is None:
            return default
        if isinstance(v, np.ndarray):
            if np.ndim(v) == 0:
                return v.item()
            return v.reshape(-1)[0].item()
        return v

    out = {}
    for k in ["grid_h","grid_w","has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
              "n_pairs_thr","n_pairs_mnn","best_inlier_ratio","best_weight_frac"]:
        out[k] = _s(z[k], 0) if k in z.files else 0

    out["grid_h"] = int(out["grid_h"]); out["grid_w"] = int(out["grid_w"])
    out["has_peak"] = int(out["has_peak"])
    out["peak_ratio"] = float(out["peak_ratio"])
    out["best_weight"] = float(out["best_weight"])
    out["best_count"] = int(out["best_count"])
    out["best_mean_sim"] = float(out["best_mean_sim"])
    out["n_pairs_thr"] = int(out["n_pairs_thr"])
    out["n_pairs_mnn"] = int(out["n_pairs_mnn"])
    out["best_inlier_ratio"] = float(out["best_inlier_ratio"])
    out["best_weight_frac"]  = float(out["best_weight_frac"])

    out["best_src"] = z["best_src"].astype(np.int32, copy=False) if "best_src" in z.files else np.zeros((0,), dtype=np.int32)
    out["best_dst"] = z["best_dst"].astype(np.int32, copy=False) if "best_dst" in z.files else np.zeros((0,), dtype=np.int32)
    out["best_sim"] = z["best_sim"].astype(np.float16, copy=False) if "best_sim" in z.files else np.zeros((0,), dtype=np.float16)
    return out

def pack_mask(mask_u8: np.ndarray) -> np.ndarray:
    m = (mask_u8 > 0).astype(np.uint8, copy=False).reshape(-1)
    return np.packbits(m, axis=None)

def _binary_open(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_opening(mask.astype(bool), structure=st).astype(np.uint8)

def _binary_close(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_closing(mask.astype(bool), structure=st).astype(np.uint8)

def _fill_holes(mask: np.ndarray) -> np.ndarray:
    if not _HAS_SCIPY:
        return mask.astype(np.uint8)
    return ndi.binary_fill_holes(mask.astype(bool)).astype(np.uint8)

def _filter_components(mask: np.ndarray, min_area: int, keep_topk: int):
    mask = mask.astype(np.uint8, copy=False)
    if mask.sum() == 0:
        return mask, 0, 0

    if not _HAS_SCIPY:
        area = int(mask.sum())
        if area < int(min_area):
            return np.zeros_like(mask, dtype=np.uint8), 0, 0
        return mask, 1, area

    lab, n = ndi.label(mask.astype(bool))
    if n == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    areas = np.bincount(lab.ravel())
    areas[0] = 0
    comps = np.where(areas >= int(min_area))[0]
    if comps.size == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    comps = comps[np.argsort(areas[comps])[::-1]]
    if keep_topk is not None and int(keep_topk) > 0:
        comps = comps[:int(keep_topk)]

    out = np.isin(lab, comps).astype(np.uint8)
    largest = int(areas[comps[0]]) if comps.size else 0
    return out, int(comps.size), largest

def _grid_morph(mask_g: np.ndarray, open_ks: int, close_ks: int):
    if (not _HAS_SCIPY) or (open_ks <= 1 and close_ks <= 1):
        return mask_g.astype(np.uint8)
    mg = mask_g.astype(bool)
    if open_ks and open_ks > 1:
        st = np.ones((open_ks, open_ks), dtype=bool)
        mg = ndi.binary_opening(mg, structure=st)
    if close_ks and close_ks > 1:
        st = np.ones((close_ks, close_ks), dtype=bool)
        mg = ndi.binary_closing(mg, structure=st)
    return mg.astype(np.uint8)

def _grid_dilate(mask_g: np.ndarray, ks: int):
    if (not _HAS_SCIPY) or ks <= 1:
        return mask_g.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_dilation(mask_g.astype(bool), structure=st).astype(np.uint8)

def grid_to_orig(mask_grid, patch, proc_h, proc_w, orig_h, orig_w):
    patch = max(int(patch), 1)
    mask_proc = np.repeat(np.repeat(mask_grid.astype(np.uint8), patch, axis=0), patch, axis=1)
    mask_proc = mask_proc[:max(int(proc_h), 1), :max(int(proc_w), 1)]
    if (int(proc_h), int(proc_w)) != (int(orig_h), int(orig_w)):
        im = Image.fromarray((mask_proc * 255).astype(np.uint8))
        im = im.resize((max(int(orig_w), 1), max(int(orig_h), 1)), resample=Image.NEAREST)
        mask_orig = (np.array(im) > 0).astype(np.uint8)
    else:
        mask_orig = mask_proc.astype(np.uint8)
    return mask_orig

def _as_save_scalar(v):
    if isinstance(v, (bool, np.bool_)):
        return np.int8(int(v))
    if isinstance(v, (int, np.integer)):
        return np.int32(int(v))
    if isinstance(v, (float, np.floating)):
        return np.float32(float(v))
    try:
        return np.float32(float(v))
    except Exception:
        return np.float32(0.0)

def save_pred_npz(out_path: Path, mask_orig: np.ndarray, grid_score: np.ndarray, grid_count: np.ndarray, scalars: dict):
    mask_pack = pack_mask(mask_orig)
    payload = dict(
        mask_pack=mask_pack.astype(np.uint8),
        mask_h=np.int32(mask_orig.shape[0]),
        mask_w=np.int32(mask_orig.shape[1]),
        grid_score=grid_score.astype(np.float16, copy=False),
        grid_count=grid_count.astype(np.uint16, copy=False),
        cfg_hash=np.array(_CFG_HASH, dtype="<U16"),
    )
    for k, v in scalars.items():
        payload[k] = _as_save_scalar(v)
    np.savez_compressed(str(out_path), **payload)

def pred_cfg_hash_matches(pred_path: Path) -> bool:
    if not pred_path.exists():
        return False
    try:
        z = np.load(pred_path, allow_pickle=False)
        if "cfg_hash" not in z.files:
            return False
        h = z["cfg_hash"]
        try:
            h = h.item()
        except Exception:
            h = str(np.array(h).reshape(-1)[0])
        return str(h) == str(_CFG_HASH)
    except Exception:
        return False

def load_pred_scalars_from_npz(pred_path: Path):
    if not pred_path.exists():
        return None
    z = np.load(pred_path, allow_pickle=False)
    keys = [
        "has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
        "inlier_ratio","pair_count","uniq_src","uniq_dst","mean_sim",
        "area_frac","n_comp","largest_comp","n_pairs_thr","n_pairs_mnn",
        "thr_used","cnt_thr_used","relaxed_used","min_pairs_used",
        "best_inlier_ratio","best_weight_frac",
        "grid_area_frac","grid_thr_q_used","grid_cnt_q_used",
        "overmask_tighten_steps",
        "feat_exists","match_exists",
    ]
    out = {}
    for k in keys:
        if k in z.files:
            v = z[k]
            out[k] = float(v) if np.ndim(v) == 0 else float(np.array(v).reshape(-1)[0])
        else:
            out[k] = 0.0
    return out

def _strong_match(mch, min_pairs_used, cfg):
    if mch is None:
        return False
    if int(mch.get("has_peak", 0)) != 1:
        return False
    bc = int(mch.get("best_count", 0))
    if bc < max(8, int(min_pairs_used) - 4):
        return False
    bms = float(mch.get("best_mean_sim", 0.0))
    pr  = float(mch.get("peak_ratio", 0.0))
    wf  = float(mch.get("best_weight_frac", 0.0))
    return (bms >= float(cfg["sim_inlier_thr"]) - 0.015) or (pr >= 1.30 and wf >= 0.10)

def build_grid_from_pairs(best_src, best_dst, best_sim, gh, gw, cfg, q_thr, q_cnt):
    N = int(gh * gw)
    if best_src is None or best_src.size == 0 or N <= 0:
        comb_grid = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
        cnt_grid  = np.zeros((max(gh,1), max(gw,1)), dtype=np.int32)
        mask_grid = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint8)
        feats = {"thr_used": 1.0, "cnt_thr_used": 999999, "grid_area_frac": 0.0}
        return comb_grid, cnt_grid, mask_grid, feats

    src = best_src.astype(np.int32, copy=False)
    dst = best_dst.astype(np.int32, copy=False)
    sim = best_sim.astype(np.float32, copy=False)

    patch_score = np.zeros((N,), dtype=np.float32)
    patch_count = np.zeros((N,), dtype=np.int32)

    w = np.clip(sim, 0.0, 1.0)
    np.add.at(patch_score, src, w); np.add.at(patch_score, dst, w)
    np.add.at(patch_count, src, 1); np.add.at(patch_count, dst, 1)

    smax = float(patch_score.max()) if patch_score.size else 0.0
    score_norm = patch_score / (smax + 1e-9) if smax > 0 else patch_score

    cmax = float(patch_count.max()) if patch_count.size else 0.0
    count_norm = patch_count.astype(np.float32) / (cmax + 1e-9) if cmax > 0 else patch_count.astype(np.float32)

    mix = float(cfg["score_mix"])
    comb = mix * score_norm + (1.0 - mix) * count_norm
    comb_grid = comb.reshape(gh, gw).astype(np.float32)
    cnt_grid  = patch_count.reshape(gh, gw).astype(np.int32)

    sig = float(cfg.get("grid_smooth_sigma", 0.0))
    if _HAS_SCIPY and sig > 0:
        comb_s = ndi.gaussian_filter(comb_grid, sigma=sig)
        mx = float(comb_s.max())
        if mx > 1e-9:
            comb_s = comb_s / mx
    else:
        comb_s = comb_grid

    comb_nz = comb_s[comb_s > 0]
    thr_dyn = float(np.quantile(comb_nz, float(q_thr))) if comb_nz.size else 1.0
    thr_used = max(float(cfg["thr_grid_floor"]), float(thr_dyn))

    cnt_nz = cnt_grid[cnt_grid > 0].astype(np.float32)
    if cnt_nz.size:
        cnt_thr = int(np.quantile(cnt_nz, float(q_cnt)))
        cnt_thr = max(1, cnt_thr)
    else:
        cnt_thr = 999999

    mask_grid = ((comb_s >= thr_used) & (cnt_grid >= cnt_thr)).astype(np.uint8)
    mask_grid = _grid_morph(mask_grid, int(cfg.get("grid_open_ks", 0) or 0), int(cfg.get("grid_close_ks", 0) or 0))

    feats = {
        "thr_used": float(thr_used),
        "cnt_thr_used": int(cnt_thr),
        "grid_area_frac": float(mask_grid.mean()) if mask_grid.size else 0.0,
    }
    return comb_s.astype(np.float32), cnt_grid.astype(np.int32), mask_grid, feats

def tighten_overmask(comb_s, cnt_grid, cfg, max_area_grid, base_q, base_cntq):
    steps_used = 0

    for q in cfg.get("tighten_q_steps", []):
        comb_nz = comb_s[comb_s > 0]
        if comb_nz.size == 0:
            break
        thr_dyn = float(np.quantile(comb_nz, float(q)))
        thr_used = max(float(cfg["thr_grid_floor"]), thr_dyn)

        cnt_nz = cnt_grid[cnt_grid > 0].astype(np.float32)
        cnt_thr = int(np.quantile(cnt_nz, float(base_cntq))) if cnt_nz.size else 999999
        cnt_thr = max(1, cnt_thr) if cnt_thr != 999999 else cnt_thr

        mg = ((comb_s >= thr_used) & (cnt_grid >= cnt_thr)).astype(np.uint8)
        mg = _grid_morph(mg, int(cfg.get("grid_open_ks", 0) or 0), int(cfg.get("grid_close_ks", 0) or 0))
        steps_used += 1
        if mg.mean() <= max_area_grid:
            return mg, float(thr_used), int(cnt_thr), steps_used

    for cq in cfg.get("tighten_cntq_steps", []):
        comb_nz = comb_s[comb_s > 0]
        if comb_nz.size == 0:
            break
        thr_dyn = float(np.quantile(comb_nz, float(base_q)))
        thr_used = max(float(cfg["thr_grid_floor"]), thr_dyn)

        cnt_nz = cnt_grid[cnt_grid > 0].astype(np.float32)
        cnt_thr = int(np.quantile(cnt_nz, float(cq))) if cnt_nz.size else 999999
        cnt_thr = max(1, cnt_thr) if cnt_thr != 999999 else cnt_thr

        mg = ((comb_s >= thr_used) & (cnt_grid >= cnt_thr)).astype(np.uint8)
        mg = _grid_morph(mg, int(cfg.get("grid_open_ks", 0) or 0), int(cfg.get("grid_close_ks", 0) or 0))
        steps_used += 1
        if mg.mean() <= max_area_grid:
            return mg, float(thr_used), int(cnt_thr), steps_used

    return None, 0.0, 0, steps_used

def select_top_components_grid(mask_g: np.ndarray, score_g: np.ndarray, keep_k: int):
    if mask_g.sum() == 0 or keep_k <= 0:
        return mask_g.astype(np.uint8), 0
    if not _HAS_SCIPY:
        return mask_g.astype(np.uint8), 1
    lab, n = ndi.label(mask_g.astype(bool))
    if n <= 1:
        return mask_g.astype(np.uint8), int(n)
    comps = np.arange(1, n+1, dtype=np.int32)
    areas = np.array([(lab == c).sum() for c in comps], dtype=np.float32)
    means = np.array([score_g[lab == c].mean() if (lab == c).any() else 0.0 for c in comps], dtype=np.float32)
    comp_score = means * np.sqrt(areas + 1e-9)
    order = np.argsort(-comp_score)
    take = comps[order[:keep_k]]
    out = np.isin(lab, take).astype(np.uint8)
    return out, int(len(take))

# ----------------------------
# Build worklists (v7.3: base on df_train_all/df_test for full coverage)
# ----------------------------
# TRAIN base
train_base_cols = [c for c in ["sample_id","case_id","variant","fold","y_forged","image_path"] if c in df_train_all.columns]
train_base = df_train_all[train_base_cols].copy()
train_base["sample_id"] = norm_id_series(train_base["sample_id"])
if "case_id" in train_base.columns:
    train_base["case_id"] = norm_id_series(train_base["case_id"])
train_base = train_base.drop_duplicates(subset=["sample_id"], keep="first")

# merge match (left -> keep all sample_id)
work_train = train_base.merge(
    man_match_train[["sample_id","match_path","match_exists"]].copy(),
    on="sample_id", how="left"
)
# merge feat manifest (left)
work_train = work_train.merge(
    man_train[["sample_id","feat_path","feat_exists"]].copy(),
    on="sample_id", how="left"
)

# TEST base
test_base = df_test[["case_id","image_path"]].copy()
test_base["case_id"] = norm_id_series(test_base["case_id"])
test_base = test_base.drop_duplicates(subset=["case_id"], keep="first")

work_test = test_base.merge(
    man_match_test[["case_id","match_path","match_exists"]].copy(),
    on="case_id", how="left"
)
work_test = work_test.merge(
    man_test[["case_id","feat_path","feat_exists"]].copy(),
    on="case_id", how="left"
)

# fill
for dfw, idc in [(work_train,"sample_id"), (work_test,"case_id")]:
    dfw[idc] = norm_id_series(dfw[idc])
    dfw["match_path"] = dfw["match_path"].fillna("").astype(str)
    dfw["feat_path"]  = dfw["feat_path"].fillna("").astype(str)
    dfw["match_exists"] = dfw["match_exists"].fillna(0).astype(int)
    dfw["feat_exists"]  = dfw["feat_exists"].fillna(0).astype(int)

# drop duplicates (hindari overwrite dobel)
work_train = work_train.drop_duplicates(subset=["sample_id"], keep="first").reset_index(drop=True)
work_test  = work_test.drop_duplicates(subset=["case_id"], keep="first").reset_index(drop=True)

# sort
sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in work_train.columns]
work_train = work_train.sort_values(sort_train_cols).reset_index(drop=True) if sort_train_cols else work_train.sort_values(["sample_id"]).reset_index(drop=True)
work_test  = work_test.sort_values(["case_id"]).reset_index(drop=True)

print("\nWork sizes:")
print("  train_all:", len(work_train), "| match_exists rate:", float(work_train["match_exists"].mean()) if len(work_train) else 0.0, "| feat_exists rate:", float(work_train["feat_exists"].mean()) if len(work_train) else 0.0)
print("  test     :", len(work_test),  "| match_exists rate:", float(work_test["match_exists"].mean())  if len(work_test)  else 0.0, "| feat_exists rate:", float(work_test["feat_exists"].mean())  if len(work_test)  else 0.0)

# ----------------------------
# Main loop
# ----------------------------
def run_pred_block(dfw: pd.DataFrame, id_col: str, out_dir: Path, label: str, collect_feat_rows: bool):
    t0 = time.time()
    done = skipped = failed = 0
    feat_rows = []

    cols = list(dfw.columns)

    for i, r in enumerate(dfw.itertuples(index=False), start=1):
        uid = str(getattr(r, id_col))
        outp = out_dir / f"{uid}.npz"

        if CFG_RECON["skip_if_exists"] and outp.exists() and pred_cfg_hash_matches(outp):
            skipped += 1
            if collect_feat_rows:
                scal = load_pred_scalars_from_npz(outp) or {}
                feat_row = {"uid": uid, **scal}
                for c in ["case_id","variant","fold","y_forged","feat_exists","match_exists"]:
                    if c in cols:
                        feat_row[c] = getattr(r, c, None)
                feat_rows.append(feat_row)
            continue

        # reset per-iter (BUGFIX: jangan nyangkut dari iterasi sebelumnya)
        comb_s = None
        cnt_g = None

        feat_path  = str(getattr(r, "feat_path", ""))
        match_path = str(getattr(r, "match_path", ""))
        ip         = str(getattr(r, "image_path", ""))

        feat_exists = int(getattr(r, "feat_exists", 0)) if "feat_exists" in cols else 0
        match_exists = int(getattr(r, "match_exists", 0)) if "match_exists" in cols else 0

        meta = _read_meta_from_feat_npz(feat_path) if feat_path else None
        mch  = _load_match_npz(match_path) if match_path else None

        # fallback meta dari image
        if meta is None:
            if ip and Path(ip).exists():
                with Image.open(ip) as im:
                    im = im.convert("RGB")
                    ow, oh = im.size
                meta = {"orig_h": int(oh), "orig_w": int(ow),
                        "proc_h": int(oh), "proc_w": int(ow),
                        "grid_h": 1, "grid_w": 1, "patch": 1}
            else:
                meta = {"orig_h": 1, "orig_w": 1, "proc_h": 1, "proc_w": 1, "grid_h": 1, "grid_w": 1, "patch": 1}

        orig_h, orig_w = int(meta["orig_h"]), int(meta["orig_w"])
        gh, gw         = int(meta["grid_h"]), int(meta["grid_w"])
        proc_h, proc_w = int(meta["proc_h"]), int(meta["proc_w"])
        patch          = int(meta["patch"])

        mask_orig = np.zeros((max(orig_h,1), max(orig_w,1)), dtype=np.uint8)

        scalars = {
            "feat_exists": int(feat_exists),
            "match_exists": int(match_exists),

            "has_peak": 0,
            "peak_ratio": 0.0,
            "best_weight": 0.0,
            "best_count": 0,
            "best_mean_sim": 0.0,
            "n_pairs_thr": 0,
            "n_pairs_mnn": 0,
            "best_inlier_ratio": 0.0,
            "best_weight_frac": 0.0,

            "inlier_ratio": 0.0,
            "pair_count": 0,
            "uniq_src": 0,
            "uniq_dst": 0,
            "mean_sim": 0.0,

            "thr_used": 0.0,
            "cnt_thr_used": 0.0,
            "relaxed_used": 0,
            "min_pairs_used": 0,

            "area_frac": 0.0,
            "n_comp": 0,
            "largest_comp": 0,

            "grid_area_frac": 0.0,
            "grid_thr_q_used": float(CFG_RECON["grid_thr_q"]),
            "grid_cnt_q_used": float(CFG_RECON["min_count_q"]),
            "overmask_tighten_steps": 0,
        }

        try:
            # align gh/gw dengan match bila ada
            if mch is not None:
                gh_m = int(mch.get("grid_h", gh))
                gw_m = int(mch.get("grid_w", gw))
                if (gh_m > 0 and gw_m > 0) and (gh_m != gh or gw_m != gw):
                    gh, gw = gh_m, gw_m

                scalars["has_peak"] = int(mch.get("has_peak", 0))
                scalars["peak_ratio"] = float(mch.get("peak_ratio", 0.0))
                scalars["best_weight"] = float(mch.get("best_weight", 0.0))
                scalars["best_count"] = int(mch.get("best_count", 0))
                scalars["best_mean_sim"] = float(mch.get("best_mean_sim", 0.0))
                scalars["n_pairs_thr"] = int(mch.get("n_pairs_thr", 0))
                scalars["n_pairs_mnn"] = int(mch.get("n_pairs_mnn", 0))
                scalars["best_inlier_ratio"] = float(mch.get("best_inlier_ratio", 0.0))
                scalars["best_weight_frac"]  = float(mch.get("best_weight_frac", 0.0))

            # allocate grid buffers AFTER final gh/gw decided
            grid_score = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
            grid_count = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint16)

            if mch is not None:
                gridN = int(gh * gw)
                min_pairs_used = int(CFG_RECON["min_pairs_small"] if gridN <= int(CFG_RECON["small_grid_N"]) else CFG_RECON["min_pairs"])
                scalars["min_pairs_used"] = int(min_pairs_used)

                if scalars["has_peak"] == 1 and scalars["best_count"] >= min_pairs_used:
                    best_src = mch.get("best_src", np.zeros((0,), dtype=np.int32))
                    best_dst = mch.get("best_dst", np.zeros((0,), dtype=np.int32))
                    best_sim = mch.get("best_sim", np.zeros((0,), dtype=np.float16))

                    q_thr = float(CFG_RECON["grid_thr_q"])
                    q_cnt = float(CFG_RECON["min_count_q"])

                    comb_s, cnt_g, mask_g, feats = build_grid_from_pairs(best_src, best_dst, best_sim, gh, gw, CFG_RECON, q_thr=q_thr, q_cnt=q_cnt)
                    scalars["thr_used"] = float(feats["thr_used"])
                    scalars["cnt_thr_used"] = float(feats["cnt_thr_used"])
                    scalars["grid_area_frac"] = float(feats["grid_area_frac"])

                    # relax if empty but strong
                    if (mask_g.sum() == 0) and bool(CFG_RECON["enable_relax_if_empty"]):
                        if _strong_match(mch, min_pairs_used, CFG_RECON):
                            q_thr2 = float(CFG_RECON["relax_q_to"])
                            q_cnt2 = float(CFG_RECON["relax_min_count_q_to"])
                            scalars["grid_thr_q_used"] = q_thr2
                            scalars["grid_cnt_q_used"] = q_cnt2
                            comb_s, cnt_g, mask_g, feats = build_grid_from_pairs(best_src, best_dst, best_sim, gh, gw, CFG_RECON, q_thr=q_thr2, q_cnt=q_cnt2)
                            scalars["thr_used"] = float(feats["thr_used"])
                            scalars["cnt_thr_used"] = float(feats["cnt_thr_used"])
                            scalars["grid_area_frac"] = float(feats["grid_area_frac"])
                            scalars["relaxed_used"] = 1

                    # convert to orig
                    mask_orig = grid_to_orig(mask_g, patch, proc_h, proc_w, orig_h, orig_w)

                    # tiny-mask assist
                    area_frac_now = float(mask_orig.sum()) / float(mask_orig.size + 1e-9)
                    if bool(CFG_RECON["enable_dilate_if_tiny"]) and area_frac_now > 0 and area_frac_now < float(CFG_RECON["tiny_area_frac"]):
                        if _strong_match(mch, min_pairs_used, CFG_RECON):
                            mask_g2 = _grid_dilate(mask_g, int(CFG_RECON["grid_dilate_ks"]))
                            mask_orig = grid_to_orig(mask_g2, patch, proc_h, proc_w, orig_h, orig_w)

                    # orig postprocess
                    if CFG_RECON["do_open"]:
                        mask_orig = _binary_open(mask_orig, int(CFG_RECON["open_ks"]))
                    if CFG_RECON["do_close"]:
                        mask_orig = _binary_close(mask_orig, int(CFG_RECON["close_ks"]))
                    if CFG_RECON.get("fill_holes", False):
                        mask_orig = _fill_holes(mask_orig)

                    # component filter
                    min_area = int(float(CFG_RECON["min_area_frac"]) * float(mask_orig.size))
                    min_area = max(1, min_area)
                    mask_orig, n_comp, largest = _filter_components(
                        mask_orig, min_area=min_area, keep_topk=int(CFG_RECON["keep_topk_components"])
                    )

                    area_frac = float(mask_orig.sum()) / float(mask_orig.size + 1e-9)

                    # overmask tighten
                    if area_frac > float(CFG_RECON["max_area_frac"]):
                        if bool(CFG_RECON["enable_tighten_overmask"]) and (comb_s is not None) and (cnt_g is not None):
                            max_area_grid = float(CFG_RECON["max_area_frac"])
                            mg_new, thr_u, cnt_u, steps_used = tighten_overmask(
                                comb_s, cnt_g, CFG_RECON,
                                max_area_grid=max_area_grid,
                                base_q=float(CFG_RECON["grid_thr_q"]),
                                base_cntq=float(CFG_RECON["min_count_q"]),
                            )
                            scalars["overmask_tighten_steps"] = int(steps_used)
                            if mg_new is not None and mg_new.sum() > 0:
                                if mg_new.mean() > max_area_grid and _HAS_SCIPY:
                                    mg_new, _ = select_top_components_grid(mg_new, comb_s, keep_k=int(CFG_RECON["keep_topk_components"]))

                                mask_orig = grid_to_orig(mg_new, patch, proc_h, proc_w, orig_h, orig_w)
                                if CFG_RECON["do_open"]:
                                    mask_orig = _binary_open(mask_orig, int(CFG_RECON["open_ks"]))
                                if CFG_RECON["do_close"]:
                                    mask_orig = _binary_close(mask_orig, int(CFG_RECON["close_ks"]))
                                if CFG_RECON.get("fill_holes", False):
                                    mask_orig = _fill_holes(mask_orig)

                                mask_orig, n_comp, largest = _filter_components(
                                    mask_orig, min_area=min_area, keep_topk=int(CFG_RECON["keep_topk_components"])
                                )
                                area_frac = float(mask_orig.sum()) / float(mask_orig.size + 1e-9)
                                if thr_u:
                                    scalars["thr_used"] = float(thr_u)
                                if cnt_u:
                                    scalars["cnt_thr_used"] = float(cnt_u)

                        # last resort: zero mask kalau masih overmask
                        if area_frac > float(CFG_RECON["max_area_frac"]):
                            mask_orig[:] = 0
                            area_frac = 0.0
                            n_comp = 0
                            largest = 0

                    # verification stats
                    simf = mch.get("best_sim", np.zeros((0,), dtype=np.float16)).astype(np.float32, copy=False)
                    inlier_ratio = float((simf >= float(CFG_RECON["sim_inlier_thr"])).mean()) if simf.size else 0.0
                    srcf = mch.get("best_src", np.zeros((0,), dtype=np.int32))
                    dstf = mch.get("best_dst", np.zeros((0,), dtype=np.int32))

                    scalars["inlier_ratio"] = float(inlier_ratio)
                    scalars["pair_count"]   = int(simf.size)
                    scalars["uniq_src"]     = int(np.unique(srcf).size) if srcf.size else 0
                    scalars["uniq_dst"]     = int(np.unique(dstf).size) if dstf.size else 0
                    scalars["mean_sim"]     = float(simf.mean()) if simf.size else 0.0
                    scalars["area_frac"]    = float(area_frac)
                    scalars["n_comp"]       = int(n_comp)
                    scalars["largest_comp"] = int(largest)

                    # store grid debug
                    if comb_s is not None:
                        grid_score = comb_s.astype(np.float32, copy=False)
                    if cnt_g is not None:
                        grid_count = cnt_g.astype(np.uint16, copy=False)

            save_pred_npz(outp, mask_orig, grid_score, grid_count, scalars)

            if collect_feat_rows:
                feat_row = {"uid": uid, **scalars}
                for c in ["case_id","variant","fold","y_forged","feat_exists","match_exists"]:
                    if c in cols:
                        feat_row[c] = getattr(r, c, None)
                feat_rows.append(feat_row)

            done += 1
            if (done > 0) and (done % int(CFG_RECON["print_every"]) == 0):
                dt = time.time() - t0
                print(f"[{label}] done={done:,} skipped={skipped:,} failed={failed:,} | {done/max(dt,1e-9):.2f} item/s | last={uid}")
                gc.collect()

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done   : {done:,}")
    print(f"  skipped: {skipped:,}")
    print(f"  failed : {failed:,}")
    print(f"  time_s : {dt:.1f}")
    return feat_rows, {"done": done, "skipped": skipped, "failed": failed, "time_s": dt}

print("\n[1/2] Build preds for TRAIN_ALL ...")
feat_rows_train, rep_train = run_pred_block(work_train, "sample_id", PRED_TRAIN_DIR, "train_all", collect_feat_rows=True)

print("\n[2/2] Build preds for TEST ...")
feat_rows_test, rep_test = run_pred_block(work_test, "case_id", PRED_TEST_DIR, "test", collect_feat_rows=False)

# ----------------------------
# Write pred manifests
# ----------------------------
man_pred_train = pd.DataFrame({
    "sample_id": work_train["sample_id"].astype(str).values,
    "case_id": work_train["case_id"].astype(str).fillna("").values if "case_id" in work_train.columns else [""]*len(work_train),
    "variant": work_train["variant"].astype(str).fillna("").values if "variant" in work_train.columns else [""]*len(work_train),
    "fold": work_train["fold"].fillna(-1).astype(int).values if "fold" in work_train.columns else [-1]*len(work_train),
    "y_forged": work_train["y_forged"].fillna(-1).astype(int).values if "y_forged" in work_train.columns else [-1]*len(work_train),
    "feat_exists": work_train["feat_exists"].fillna(0).astype(int).values,
    "match_exists": work_train["match_exists"].fillna(0).astype(int).values,
    "pred_path": [str(PRED_TRAIN_DIR / f"{sid}.npz") for sid in work_train["sample_id"].astype(str).values],
})
man_pred_train["pred_exists"] = man_pred_train["pred_path"].map(lambda p: Path(p).exists()).astype(int)

man_pred_test = pd.DataFrame({
    "case_id": work_test["case_id"].astype(str).values,
    "feat_exists": work_test["feat_exists"].fillna(0).astype(int).values,
    "match_exists": work_test["match_exists"].fillna(0).astype(int).values,
    "pred_path": [str(PRED_TEST_DIR / f"{cid}.npz") for cid in work_test["case_id"].astype(str).values],
})
man_pred_test["pred_exists"] = man_pred_test["pred_path"].map(lambda p: Path(p).exists()).astype(int)

PRED_MAN_TRAIN_PATH = PRED_ROOT / "manifest_pred_train_all.csv"
PRED_MAN_TEST_PATH  = PRED_ROOT / "manifest_pred_test.csv"
man_pred_train.to_csv(PRED_MAN_TRAIN_PATH, index=False)
man_pred_test.to_csv(PRED_MAN_TEST_PATH, index=False)

df_pred_feat_train_all = pd.DataFrame(feat_rows_train) if len(feat_rows_train) else pd.DataFrame()
PRED_FEAT_TRAIN_PATH = PRED_ROOT / "pred_features_train_all.csv"
df_pred_feat_train_all.to_csv(PRED_FEAT_TRAIN_PATH, index=False)

with open(PRED_ROOT / "pred_summary.json", "w") as f:
    json.dump({
        "cfg": CFG_RECON,
        "cfg_hash": _CFG_HASH,
        "MAN_TRAIN_PATH": str(MAN_TRAIN_PATH),
        "MAN_TEST_PATH": str(MAN_TEST_PATH),
        "MATCH_MAN_TRAIN_PATH": str(MATCH_MAN_TRAIN_PATH),
        "MATCH_MAN_TEST_PATH": str(MATCH_MAN_TEST_PATH),
        "train": rep_train,
        "test": rep_test,
        "outputs": {
            "PRED_MAN_TRAIN_PATH": str(PRED_MAN_TRAIN_PATH),
            "PRED_MAN_TEST_PATH": str(PRED_MAN_TEST_PATH),
            "PRED_FEAT_TRAIN_PATH": str(PRED_FEAT_TRAIN_PATH),
        }
    }, f, indent=2)

print("\nWrote pred manifests:")
print(" -", PRED_MAN_TRAIN_PATH, "| exists_rate:", float(man_pred_train["pred_exists"].mean()) if len(man_pred_train) else 0.0)
print(" -", PRED_MAN_TEST_PATH,  "| exists_rate:", float(man_pred_test["pred_exists"].mean()) if len(man_pred_test) else 0.0)
print("Extra:")
print(" -", PRED_FEAT_TRAIN_PATH)
print(" -", PRED_ROOT / "pred_summary.json")

PRED_CACHE_ROOT = str(PRED_ROOT)
PRED_TRAIN_CACHE_DIR = str(PRED_TRAIN_DIR)
PRED_TEST_CACHE_DIR  = str(PRED_TEST_DIR)

print("\nDONE. Exported:")
print("- CFG_RECON, PRED_CACHE_ROOT, PRED_TRAIN_CACHE_DIR, PRED_TEST_CACHE_DIR")
print("- PRED_MAN_TRAIN_PATH, PRED_MAN_TEST_PATH, PRED_FEAT_TRAIN_PATH")
print("\nTrain feature head:")
print(df_pred_feat_train_all.head())

globals().update({
    "CFG_RECON": CFG_RECON,
    "PRED_ROOT": PRED_ROOT,
    "PRED_CACHE_ROOT": PRED_CACHE_ROOT,
    "PRED_TRAIN_CACHE_DIR": PRED_TRAIN_CACHE_DIR,
    "PRED_TEST_CACHE_DIR": PRED_TEST_CACHE_DIR,
    "PRED_MAN_TRAIN_PATH": PRED_MAN_TRAIN_PATH,
    "PRED_MAN_TEST_PATH": PRED_MAN_TEST_PATH,
    "PRED_FEAT_TRAIN_PATH": PRED_FEAT_TRAIN_PATH,
})
